# FBL contract-driven data-quality pipeline
contracts > bronze > silver (+DQ) > gold (+assertions) > semantic (metric views + contracts) > report.
One notebook; set the `stage` widget (or run `all`). See docs/DQ_PIPELINE.md and docs/DATA_CONTRACTS.md.
Generated by tools/build_fbl_notebook.py. Edit src/lib/*.py or contracts/ and rebuild instead of editing this file.


In [ ]:
"""Rules-driven data-quality engine (PySpark).

A RULE is one row of governance.dq_rules (source: data/dq/dq_rules.csv):
  rule_id, layer, table_name, rule_type, columns, expression, action, description, owner, active

rule_type    what it checks (a row VIOLATES the rule when...)
  not_null          any listed column is NULL or blank
  allowed_values    column is not in the list in `expression`, e.g. 'L','R','B'
  range             column is outside `expression`, written like  60 AND 84
  custom            the SQL condition in `expression` is FALSE  (business rules). NULL = not a violation
                    (NULLs are caught by a not_null rule instead, so one problem is reported once)
  referential       column value not found in another table: expression = schema.table.column
  exact_duplicate   row is an identical copy of an earlier row (all data columns equal)
  unique            key columns repeat: later copies violate (the first copy is kept)
  unique_strict     key columns repeat: EVERY copy violates (conflicting duplicates need a human)
  standardized      value was changed by silver cleaning (e.g. 'r' -> 'R'): logged, never rejected
  no_nulls          gold guarantee: no NULL in any column (columns '*' minus `expression` exclusions)

action       what happens to violating rows
  dedupe       auto-corrected: extra exact copies removed (disposition = auto_corrected)
  standardize  auto-corrected value kept, logged (disposition = auto_corrected)
  quarantine   removed from the clean table, written to governance.dq_quarantine (disposition = rejected)
  warn         row stays, violation logged (disposition = warning)
  fail         the stage raises an error after logging, stopping the job

Conventions: the DataFrame has `_row_id` (unique per row) and optionally `_raw` (JSON of the original bronze row).
Columns starting with "_" are metadata, never checked as data.
"""
import csv
import io
from functools import reduce

from pyspark.sql import DataFrame, SparkSession, Window, functions as F

RULE_TYPES = {"not_null", "allowed_values", "range", "custom", "referential", "exact_duplicate",
              "unique", "unique_strict", "standardized", "no_nulls"}
ACTIONS = {"dedupe", "standardize", "quarantine", "warn", "fail"}

RESULTS_DDL = ("run_id STRING, checked_at TIMESTAMP, layer STRING, table_name STRING, rule_id STRING, "
               "rule_type STRING, action STRING, violations BIGINT, rows_checked BIGINT, description STRING")
STATS_DDL = ("run_id STRING, checked_at TIMESTAMP, layer STRING, table_name STRING, rows_in BIGINT, "
             "rows_deduped BIGINT, rows_standardized BIGINT, rows_warned BIGINT, rows_rejected BIGINT, rows_out BIGINT")
QUARANTINE_DDL = ("run_id STRING, checked_at TIMESTAMP, source_table STRING, row_id BIGINT, rule_ids STRING, "
                  "disposition STRING, payload STRING, status STRING, fix_notes STRING")
RULES_DDL = ("rule_id STRING, layer STRING, table_name STRING, rule_type STRING, columns STRING, expression STRING, "
             "action STRING, description STRING, owner STRING, active STRING")


class DQFailure(Exception):
    """Raised when a rule with action 'fail' (or any gold assertion) is violated."""


# --------------------------------------------------------------------------- rules
def parse_rules_csv(text: str) -> list:
    rules = [{k: (v or "").strip() for k, v in row.items()} for row in csv.DictReader(io.StringIO(text))]
    validate_rules(rules)
    return rules


def validate_rules(rules: list):
    seen = set()
    for r in rules:
        if r["rule_id"] in seen:
            raise ValueError(f"duplicate rule_id {r['rule_id']}")
        seen.add(r["rule_id"])
        if r["rule_type"] not in RULE_TYPES:
            raise ValueError(f"{r['rule_id']}: unknown rule_type {r['rule_type']!r}")
        if r["action"] not in ACTIONS:
            raise ValueError(f"{r['rule_id']}: unknown action {r['action']!r}")
        if r["rule_type"] in {"allowed_values", "range", "custom", "referential"} and not r["expression"]:
            raise ValueError(f"{r['rule_id']}: {r['rule_type']} needs an expression")
        if r["rule_type"] != "exact_duplicate" and not r["columns"] and r["rule_type"] != "custom":
            raise ValueError(f"{r['rule_id']}: columns required")
        if r["rule_type"] == "exact_duplicate" and r["action"] != "dedupe":
            raise ValueError(f"{r['rule_id']}: exact_duplicate must use action dedupe")


def load_rules(spark: SparkSession, layer: str, table_name: str) -> list:
    rows = (spark.table("governance.dq_rules")
            .where((F.col("layer") == layer) & (F.col("table_name") == table_name) & (F.col("active") == "Y"))
            .orderBy("rule_id").collect())
    return [r.asDict() for r in rows]


# --------------------------------------------------------------------------- helpers
def data_columns(df: DataFrame) -> list:
    return [c for c in df.columns if not c.startswith("_")]


def _cols(rule: dict) -> list:
    return [c.strip() for c in (rule.get("columns") or "").split(",") if c.strip()]


def _q(c: str) -> str:
    return f"`{c}`"


def _predicate(rule: dict, df_cols: list):
    t, cols, e = rule["rule_type"], _cols(rule), (rule.get("expression") or "").strip()
    if t == "not_null":
        return F.expr(" OR ".join(f"({_q(c)} IS NULL OR trim(CAST({_q(c)} AS STRING)) = '')" for c in cols))
    if t == "allowed_values":
        return F.expr(f"{_q(cols[0])} IS NOT NULL AND {_q(cols[0])} NOT IN ({e})")
    if t == "range":
        return F.expr(f"{_q(cols[0])} IS NOT NULL AND NOT ({_q(cols[0])} BETWEEN {e})")
    if t == "custom":
        return F.expr(f"NOT coalesce(({e}), TRUE)")
    if t == "no_nulls":
        exclude = {x.strip() for x in e.split(",") if x.strip()}
        check = [c for c in df_cols if c not in exclude] if cols == ["*"] else cols
        return F.expr(" OR ".join(f"{_q(c)} IS NULL" for c in check))
    if t == "standardized":
        c = cols[0]
        raw = f"get_json_object(_raw, '$.{c}')"
        return F.expr(f"{raw} IS NOT NULL AND {_q(c)} IS NOT NULL AND {raw} <> CAST({_q(c)} AS STRING)")
    raise ValueError(f"no predicate for {t}")


def _add_flag(spark, df: DataFrame, rule: dict, flag: str) -> DataFrame:
    t, cols = rule["rule_type"], _cols(rule)
    if t in ("unique", "unique_strict"):
        w = Window.partitionBy(*[F.col(_q(k)) for k in cols])
        dup = (F.row_number().over(w.orderBy("_row_id")) > 1) if t == "unique" else (F.count(F.lit(1)).over(w) > 1)
        keys_present = reduce(lambda a, b: a & b, [F.col(_q(k)).isNotNull() for k in cols])
        return df.withColumn(flag, F.coalesce(dup & keys_present, F.lit(False)))
    if t == "referential":
        ref_table, ref_col = rule["expression"].rsplit(".", 1)
        ref = spark.table(ref_table).select(F.col(ref_col).alias("_ref_k")).distinct()
        col = cols[0]
        return (df.join(ref, df[_q(col)] == F.col("_ref_k"), "left")
                  .withColumn(flag, F.col(_q(col)).isNotNull() & F.col("_ref_k").isNull())
                  .drop("_ref_k"))
    return df.withColumn(flag, F.coalesce(_predicate(rule, [c for c in df.columns if not c.startswith("_")]), F.lit(False)))


def _log_frame(df: DataFrame, run_id: str, table_label: str, rule_ids, disposition: str) -> DataFrame:
    payload = F.col("_raw") if "_raw" in df.columns else F.to_json(F.struct(*[F.col(_q(c)) for c in data_columns(df)]))
    rids = rule_ids if not isinstance(rule_ids, str) else F.lit(rule_ids)
    return df.select(
        F.lit(run_id).alias("run_id"), F.current_timestamp().alias("checked_at"), F.lit(table_label).alias("source_table"),
        F.col("_row_id").alias("row_id"), rids.alias("rule_ids"), F.lit(disposition).alias("disposition"),
        payload.alias("payload"), F.lit("open").alias("status"), F.lit(None).cast("string").alias("fix_notes"))


def _empty(spark, ddl):
    return spark.createDataFrame([], ddl)


# --------------------------------------------------------------------------- main entry point
def apply_rules(spark: SparkSession, df: DataFrame, rules: list, run_id: str, layer: str, table_label: str,
                mode: str = "filter"):
    """Apply rules to df.

    mode='filter': remove deduped/quarantined rows. Returns (clean_df, log_df, results_df, stats_dict).
    mode='assert': never remove rows (use for gold); every violation is treated as a failure.
    """
    assert mode in ("filter", "assert")
    if "_row_id" not in df.columns:      # e.g. gold tables: give every row a stable id for ordering/logging
        df = df.withColumn("_row_id", F.monotonically_increasing_id())
    log_frames, results = [], []
    rows_in = df.count()
    deduped = 0

    def result(r, n, checked):
        results.append((run_id, None, layer, table_label, r["rule_id"], r["rule_type"],
                        "fail" if mode == "assert" else r["action"], int(n), int(checked), r["description"]))

    # pass 1: exact duplicates are auto-corrected BEFORE key uniqueness is judged
    for r in [x for x in rules if x["action"] == "dedupe"]:
        w = Window.partitionBy(*[F.col(_q(c)) for c in data_columns(df)]).orderBy("_row_id")
        d = df.withColumn("_dup", F.row_number().over(w) > 1)
        extra = d.where("_dup")
        n = extra.count()
        if n and mode == "filter":
            log_frames.append(_log_frame(extra, run_id, table_label, r["rule_id"], "auto_corrected"))
            df = d.where(~F.col("_dup")).drop("_dup")
            deduped += n
        result(r, n, rows_in)

    after_dedupe = df.count()
    flagged, flags = df, []
    for i, r in enumerate([x for x in rules if x["action"] != "dedupe"]):
        flag = f"_v{i}"
        flagged = _add_flag(spark, flagged, r, flag)
        flags.append((r, flag))

    counts = flagged.agg(*[F.sum(F.col(f).cast("int")).alias(f) for _, f in flags]).first() if flags else None
    for r, f in flags:
        result(r, (counts[f] or 0), after_dedupe)

    flag_cols = [f for _, f in flags]
    quar = [(r, f) for r, f in flags if r["action"] == "quarantine"]
    reasons = F.concat_ws(";", *[F.when(F.col(f), F.lit(r["rule_id"])) for r, f in quar]) if quar else F.lit("")
    flagged = flagged.withColumn("_reasons", reasons)

    rejected = standardized = warned = 0
    if mode == "filter":
        bad = flagged.where(F.col("_reasons") != "")
        rejected = bad.count()
        if rejected:
            log_frames.append(_log_frame(bad, run_id, table_label, F.col("_reasons"), "rejected"))
        for r, f in flags:
            if r["action"] == "standardize":
                s = flagged.where(F.col(f))
                standardized += s.count()
                log_frames.append(_log_frame(s, run_id, table_label, r["rule_id"], "auto_corrected"))
            elif r["action"] == "warn":
                wdf = flagged.where(F.col(f))
                warned += wdf.count()
                log_frames.append(_log_frame(wdf, run_id, table_label, r["rule_id"], "warning"))
        clean = flagged.where(F.col("_reasons") == "").drop(*flag_cols, "_reasons")
    else:
        clean = flagged.drop(*flag_cols, "_reasons")

    rows_out = clean.count()
    log_df = reduce(DataFrame.unionByName, log_frames) if log_frames else _empty(spark, QUARANTINE_DDL)
    results_df = (spark.createDataFrame(results, RESULTS_DDL.replace("checked_at TIMESTAMP", "checked_at STRING"))
                  .withColumn("checked_at", F.current_timestamp()) if results else _empty(spark, RESULTS_DDL))
    results_df = results_df.select(*[c.split()[0] for c in RESULTS_DDL.split(", ")])
    stats = dict(run_id=run_id, layer=layer, table_name=table_label, rows_in=rows_in, rows_deduped=deduped,
                 rows_standardized=standardized, rows_warned=warned, rows_rejected=rejected, rows_out=rows_out)
    return clean, log_df, results_df, stats


def failures(results_df: DataFrame) -> list:
    """Rules with action 'fail' that were violated."""
    return [r.asDict() for r in results_df.where("action = 'fail' AND violations > 0").collect()]


def raise_if_failed(results_df: DataFrame):
    bad = failures(results_df)
    if bad:
        raise DQFailure("Data quality failure: " + "; ".join(
            f"{b['table_name']} {b['rule_id']} ({b['violations']}): {b['description']}" for b in bad))


In [ ]:
"""Data contracts: load, lint, compile, bundle, diff.  Pure Python (no Spark), so it runs in CI, locally and in Databricks.

Contracts are YAML modeled on the Open Data Contract Standard v3.1 (apiVersion/kind/version, schema[] objects with
properties and nested quality, slaProperties, extensions ONLY via customProperties). Anything specific to this project lives
in customProperties or in `type: custom` quality rules, so the files stay valid ODCS-shaped documents. Verify field names
against the ODCS version your organisation standardises on; this module is the single place that knows the mapping.

Contract layers (customProperties.layer):
  source    the file/table as delivered (bronze).  properties = columns that must be present, all strings
  silver    typed + cleaned.                       drives typed SELECT generation and silver DQ rules
  gold      the published star schema.             drives schema conformance and gold assertions
  semantic  a metric view.                         interface (fields/measures), golden tests, SLA, definition hash

Runtime flow:  YAML (git) -> lint -> bundle JSON  -> publish to governance.data_contracts -> pipeline reads the registry.
"""
import glob
import hashlib
import json
import os
import re

try:                                    # PyYAML is optional at runtime: Databricks can consume the JSON bundle instead
    import yaml
except ImportError:                     # pragma: no cover
    yaml = None

LAYERS = {"source", "silver", "gold", "semantic"}
LOGICAL_TYPES = {"string", "integer", "number", "boolean", "date", "timestamp"}
STATUSES = {"draft", "active", "deprecated"}
ENGINE_DQ = "fbl-dq-engine"
ENGINE_SEM = "fbl-semantic-checks"
SEMVER = re.compile(r"^\d+\.\d+\.\d+$")
RULE_TYPES = {"not_null", "allowed_values", "range", "custom", "referential", "exact_duplicate", "unique", "unique_strict",
              "standardized", "no_nulls"}
ACTIONS = {"dedupe", "standardize", "quarantine", "warn", "fail"}
TEST_TYPES = {"reconcile", "expect", "bounds"}
STRICTNESS = {"fail": 4, "quarantine": 3, "dedupe": 2, "standardize": 1, "warn": 0}


# --------------------------------------------------------------------------- loading
def parse_yaml(text: str) -> dict:
    if yaml is None:
        raise ImportError("PyYAML is not installed. In a notebook run `%pip install pyyaml`, or upload the JSON bundle "
                          "(python tools/contract_tool.py bundle) instead of the YAML files.")
    return yaml.safe_load(text)


def checksum(contract: dict) -> str:
    return hashlib.sha256(json.dumps(contract, sort_keys=True).encode()).hexdigest()[:16]


def make_bundle(contracts: list, definitions: dict = None) -> dict:
    """One JSON file holding every contract plus the metric-view SQL they point at: the easiest thing to upload by hand."""
    return {"bundleVersion": 1, "contracts": [{"id": c["id"], "checksum": checksum(c), "contract": c} for c in contracts],
            "definitions": definitions or {}}


def discover_contracts(directory: str):
    """Find contracts in a folder (e.g. a Volume): contracts_bundle.json wins; otherwise *.yaml files (needs PyYAML).
    Returns (contracts, definitions {basename: sql text}, origin) or None if nothing is there."""
    bpath = os.path.join(directory, "contracts_bundle.json")
    if os.path.exists(bpath):
        with open(bpath) as fh:
            b = json.load(fh)
        return contracts_from_bundle(b), b.get("definitions", {}), f"bundle {bpath}"
    files = sorted(glob.glob(os.path.join(directory, "**", "*.y*ml"), recursive=True))
    if not files:
        return None
    contracts = []
    for f in files:
        with open(f) as fh:
            contracts.append(parse_yaml(fh.read()))
    defs = {}
    for f in glob.glob(os.path.join(directory, "**", "definitions", "*.sql"), recursive=True):
        with open(f) as fh:
            defs[os.path.basename(f)] = fh.read()
    return contracts, defs, f"{len(files)} yaml files in {directory}"


def contracts_from_bundle(bundle) -> list:
    if isinstance(bundle, str):
        bundle = json.loads(bundle)
    return [b["contract"] for b in bundle["contracts"]]


def cprops(obj: dict) -> dict:
    """customProperties list -> dict."""
    return {p["property"]: p.get("value") for p in (obj or {}).get("customProperties", []) or []}


def layer_of(c: dict) -> str:
    return cprops(c).get("layer", "")


def main_object(c: dict) -> dict:
    return c["schema"][0]


def render(text: str, catalog: str) -> str:
    return re.sub(r"\{\{\s*catalog\s*\}\}", catalog, text)


class ContractSet:
    def __init__(self, contracts: list):
        self.contracts = list(contracts)
        self.by_id = {c["id"]: c for c in contracts}

    def layer(self, layer: str, product: str = None) -> list:
        return [c for c in self.contracts if layer_of(c) == layer and (product is None or c.get("dataProduct") == product)]

    def entity(self, layer: str, entity: str, product: str = None):
        for c in self.layer(layer, product):
            if cprops(main_object(c)).get("entity") == entity:
                return c
        return None

    def __len__(self):
        return len(self.contracts)


# --------------------------------------------------------------------------- compile
def compile_rules(contract: dict) -> list:
    """Silver/gold contract -> rows for governance.dq_rules (same shape the engine already consumes)."""
    layer = layer_of(contract)
    if layer not in ("silver", "gold"):
        return []
    obj = main_object(contract)
    entity = cprops(obj)["entity"]
    rules = []
    for q in obj.get("quality", []) or []:
        if q.get("type") == "custom" and q.get("engine") == ENGINE_DQ:
            i = q["implementation"]
            cols = i.get("columns", "")
            rules.append({
                "rule_id": q["name"], "layer": layer, "table_name": entity, "rule_type": i["rule_type"],
                "columns": ",".join(cols) if isinstance(cols, list) else str(cols),
                "expression": str(i.get("expression", "") or ""), "action": i["action"],
                "description": q.get("description", ""), "owner": i.get("owner", "Data Engineering"), "active": "Y"})
    return rules


def compile_all_rules(cs: ContractSet) -> list:
    return [r for c in cs.contracts for r in compile_rules(c)]


def expected_columns(source_contract: dict) -> list:
    return [p["name"] for p in main_object(source_contract)["properties"]]


def typed_select(silver: dict, source: dict) -> str:
    """Generate the silver typing SELECT from the contracts: trim, blank -> NULL, standardize, try_cast."""
    cast = {"integer": "INT", "number": "DOUBLE", "date": "DATE", "boolean": "BOOLEAN", "timestamp": "TIMESTAMP"}
    sobj, bobj = main_object(silver), main_object(source)
    cols = []
    for p in sobj["properties"]:
        cp = cprops(p)
        src = cp.get("sourceColumn", p["name"])
        base = f"NULLIF(trim(`{src}`), '')"
        t = p["logicalType"]
        if t == "string":
            std = cp.get("standardize")
            expr = {"upper": f"upper({base})", "lower": f"lower({base})", "initcap": f"initcap({base})"}.get(std, base)
        else:
            expr = f"try_cast({base} AS {cast[t]})"
        cols.append(f"{expr} AS `{p['name']}`")
    raw = ", ".join(f"t.`{p['name']}`" for p in bobj["properties"])
    cols += ["_row_id", f"to_json(struct({raw})) AS _raw", "_ingested_at"]
    return "SELECT\n  " + ",\n  ".join(cols) + f"\nFROM {bobj['physicalName']} t"


# --------------------------------------------------------------------------- metric view definitions (for semantic contracts)
def load_metric_view_definition(sql_text: str, catalog: str = "cat") -> dict:
    """Pull the YAML out of a CREATE VIEW ... WITH METRICS ... AS $$ yaml $$ file."""
    parts = render(sql_text, catalog).split("$$")
    if len(parts) < 3:
        raise ValueError("no $$ ... $$ YAML block found")
    return parse_yaml(parts[1])


def expr_hash(expr: str) -> str:
    return hashlib.sha256(" ".join(str(expr).split()).encode()).hexdigest()[:12]


def definition_hashes(spec: dict) -> dict:
    out = {}
    for f in spec.get("fields", spec.get("dimensions", [])) or []:
        out[f["name"]] = expr_hash(f["expr"])
    for m in spec.get("measures", []) or []:
        out[m["name"]] = expr_hash(m["expr"])
    return out


def stamp_expr_hashes(yaml_text: str, spec: dict) -> str:
    """Rewrite each property's exprHash in a semantic contract (text patch, keeps comments and layout)."""
    for name, h in definition_hashes(spec).items():
        block = re.compile(rf"(\n[ ]*- name: {re.escape(name)}\n(?:(?![ ]*- name: ).*\n)*?[ ]*- property: exprHash\n[ ]*value: )[0-9a-f]+")
        yaml_text = block.sub(lambda m: m.group(1) + h, yaml_text, count=1)
    return yaml_text


# --------------------------------------------------------------------------- lint
def lint(cs: ContractSet, definitions: dict = None) -> list:
    """Return [(severity, contract_id, message)]. severity is 'error' or 'warn'. definitions: {path: parsed metric-view spec}."""
    out, rule_ids = [], {}
    def err(c, m): out.append(("error", c.get("id", "?"), m))
    def warn(c, m): out.append(("warn", c.get("id", "?"), m))

    ids = [c.get("id") for c in cs.contracts]
    for i in {x for x in ids if ids.count(x) > 1}:
        out.append(("error", i, "duplicate contract id"))

    for c in cs.contracts:
        for k in ("apiVersion", "kind", "id", "version", "status", "schema", "dataProduct"):
            if k not in c:
                err(c, f"missing required key {k!r}")
        if "schema" not in c or not c["schema"]:
            continue
        cp = cprops(c)
        layer = cp.get("layer", "")
        if c.get("kind") != "DataContract":
            err(c, "kind must be DataContract")
        if layer not in LAYERS:
            err(c, f"customProperties.layer must be one of {sorted(LAYERS)}"); continue
        if c.get("status") not in STATUSES:
            err(c, f"status must be one of {sorted(STATUSES)}")
        if not SEMVER.match(str(c.get("version", ""))):
            err(c, "version must be semantic (1.2.3)")
        if c.get("status") == "deprecated" and not cp.get("replacedBy"):
            warn(c, "deprecated contract should name customProperties.replacedBy")
        if not (c.get("description") or {}).get("purpose"):
            warn(c, "description.purpose is empty")
        if not cp.get("owner"):
            err(c, "customProperties.owner is required")
        parts = str(c["id"]).split(".")
        if len(parts) != 3 or parts[0] != c.get("dataProduct") or parts[1] != layer:
            err(c, "id must look like <dataProduct>.<layer>.<name>")

        obj = main_object(c)
        ocp = cprops(obj)
        if not obj.get("physicalName"):
            err(c, "schema[0].physicalName is required")
        names = [p.get("name") for p in obj.get("properties", [])]
        if not names:
            err(c, "schema[0].properties is empty")
        if len(names) != len(set(names)):
            err(c, "duplicate property names")
        for p in obj.get("properties", []):
            if p.get("logicalType") not in LOGICAL_TYPES:
                err(c, f"property {p.get('name')}: logicalType must be one of {sorted(LOGICAL_TYPES)}")
            if not p.get("description"):
                warn(c, f"property {p.get('name')}: missing description")

        # quality rules
        rules = []
        for q in obj.get("quality", []) or []:
            if q.get("type") == "custom" and q.get("engine") == ENGINE_DQ:
                i = q.get("implementation") or {}
                rid = q.get("name", "")
                if not rid:
                    err(c, "quality rule without name (rule id)"); continue
                if rid in rule_ids:
                    err(c, f"rule id {rid} also used in {rule_ids[rid]}")
                rule_ids[rid] = c["id"]
                if i.get("rule_type") not in RULE_TYPES:
                    err(c, f"{rid}: unknown rule_type {i.get('rule_type')!r}")
                if i.get("action") not in ACTIONS:
                    err(c, f"{rid}: unknown action {i.get('action')!r}")
                if layer == "gold" and i.get("action") != "fail":
                    err(c, f"{rid}: gold rules must use action 'fail'")
                if i.get("rule_type") in {"allowed_values", "range", "custom", "referential"} and not i.get("expression"):
                    err(c, f"{rid}: {i.get('rule_type')} needs an expression")
                if not q.get("description"):
                    warn(c, f"{rid}: missing description")
                rules.append(i)
                cols = i.get("columns", "")
                cols = cols if isinstance(cols, list) else [x.strip() for x in str(cols).split(",") if x.strip()]
                for col in cols:
                    if col != "*" and col not in names:
                        err(c, f"{rid}: column {col!r} is not a property of this schema")
            elif q.get("type") == "custom" and q.get("engine") == ENGINE_SEM:
                if layer != "semantic":
                    err(c, f"{q.get('name')}: semantic tests belong in semantic contracts")
            elif q.get("type") == "library" and q.get("metric") == "rowCount":
                pass
            else:
                warn(c, f"quality item {q.get('name') or q.get('metric') or '?'} is not enforced by this toolchain")

        if layer in ("silver", "gold"):
            if not ocp.get("entity"):
                err(c, "schema[0].customProperties.entity is required")
            covered_nn = set()
            keys_covered = []
            for i in rules:
                cols = i.get("columns", "")
                cols = cols if isinstance(cols, list) else [x.strip() for x in str(cols).split(",") if x.strip()]
                if i.get("rule_type") == "not_null":
                    covered_nn |= set(cols)
                if i.get("rule_type") == "no_nulls":
                    covered_nn |= set(names) if cols == ["*"] or not cols else set(cols)
                if i.get("rule_type") in ("unique", "unique_strict"):
                    keys_covered.append(set(cols))
            for p in obj.get("properties", []):
                if p.get("required") and p["name"] not in covered_nn:
                    err(c, f"property {p['name']} is required but no not_null/no_nulls rule covers it")
                if not p.get("required") and p["name"] in covered_nn:
                    warn(c, f"property {p['name']} is checked as non-null but not marked required")
            pk = {p["name"] for p in obj.get("properties", []) if p.get("primaryKey")}
            if pk and pk not in keys_covered:
                err(c, f"primary key {sorted(pk)} has no matching unique/unique_strict rule")
            for p in obj.get("properties", []):
                for r in p.get("relationships", []) or []:
                    has_ref = any(i.get("rule_type") == "referential" and p["name"] in str(i.get("columns", "")) for i in rules)
                    if not has_ref:
                        err(c, f"property {p['name']} has a foreignKey relationship but no referential rule")
            if layer == "silver":
                src_id = cp.get("dependsOn") or ocp.get("dependsOn")
                src = cs.by_id.get(src_id)
                if not src:
                    err(c, f"dependsOn {src_id!r} is not a known contract")
                else:
                    src_names = {p["name"] for p in main_object(src)["properties"]}
                    for p in obj.get("properties", []):
                        sc = cprops(p).get("sourceColumn", p["name"])
                        if sc not in src_names:
                            err(c, f"property {p['name']}: source column {sc!r} not in {src_id}")
            if layer == "gold" and ocp.get("reconcilesTo") and not ocp["reconcilesTo"].startswith("silver."):
                err(c, "reconcilesTo should name a silver table")
        if layer == "source":
            for p in obj.get("properties", []):
                if p.get("logicalType") != "string":
                    err(c, f"source property {p['name']} must be string (typing happens in silver)")
            if not cprops(obj).get("landingFolder"):
                err(c, "source schema needs customProperties.landingFolder")

        if layer == "semantic":
            kinds = {p["name"]: cprops(p).get("kind") for p in obj.get("properties", [])}
            for n, k in kinds.items():
                if k not in ("field", "measure"):
                    err(c, f"property {n}: customProperties.kind must be field or measure")
            if not ocp.get("definition"):
                err(c, "schema[0].customProperties.definition (path to the metric view SQL) is required")
            if not any(s.get("property") == "freshness" for s in c.get("slaProperties", []) or []):
                warn(c, "no freshness SLA declared")
            for dep in ocp.get("dependsOn", []) or []:
                if dep not in cs.by_id and dep.split(".")[0] == c.get("dataProduct"):
                    warn(c, f"dependsOn {dep} has no contract")
            measures = {n for n, k in kinds.items() if k == "measure"}
            fields = {n for n, k in kinds.items() if k == "field"}
            for q in obj.get("quality", []) or []:
                if q.get("engine") != ENGINE_SEM:
                    continue
                i = q.get("implementation") or {}
                if i.get("test_type") not in TEST_TYPES:
                    err(c, f"{q.get('name')}: test_type must be one of {sorted(TEST_TYPES)}"); continue
                for d in i.get("dimensions", []) or []:
                    if d not in fields:
                        err(c, f"{q.get('name')}: dimension {d!r} is not a field of this contract")
                ms = list(i.get("bounds", {}).keys()) if i["test_type"] == "bounds" else (i.get("measures", []) or [])
                for m in ms:
                    if m not in measures:
                        err(c, f"{q.get('name')}: measure {m!r} is not a measure of this contract")
                if i["test_type"] == "reconcile" and not i.get("compare_sql"):
                    err(c, f"{q.get('name')}: reconcile needs compare_sql")
                if i["test_type"] == "expect" and not i.get("expect"):
                    err(c, f"{q.get('name')}: expect needs expected rows")
            if definitions is not None:
                spec = definitions.get(ocp.get("definition")) or definitions.get(os.path.basename(str(ocp.get("definition"))))
                if spec is None:
                    err(c, f"definition file {ocp.get('definition')!r} not found")
                else:
                    live = definition_hashes(spec)
                    for n in kinds:
                        if n not in live:
                            err(c, f"{n} is in the contract but not in the metric view definition (breaking removal)")
                    for n in live:
                        if n not in kinds:
                            warn(c, f"{n} exists in the metric view but is not in the contract (undocumented)")
                    for p in obj.get("properties", []):
                        want = cprops(p).get("exprHash")
                        if p["name"] in live and want and want != live[p["name"]]:
                            err(c, f"{p['name']}: definition changed (hash {live[p['name']]} != contract {want}). "
                                   "Bump the contract version (major if consumers are affected) and run `contract_tool.py stamp`")
    return out


# --------------------------------------------------------------------------- diff (breaking-change detection)
def diff_contracts(old: dict, new: dict) -> dict:
    """Compare two versions of the SAME contract. Returns {'breaking': [...], 'non_breaking': [...], 'policy_errors': [...]}."""
    br, nb, pol = [], [], []
    oo, no = main_object(old), main_object(new)
    op = {p["name"]: p for p in oo.get("properties", [])}
    np_ = {p["name"]: p for p in no.get("properties", [])}
    for n in op:
        if n not in np_:
            br.append(f"property removed: {n}")
    for n in np_:
        if n not in op:
            nb.append(f"property added: {n}")
    for n in op.keys() & np_.keys():
        a, b = op[n], np_[n]
        if a.get("logicalType") != b.get("logicalType"):
            br.append(f"{n}: type changed {a.get('logicalType')} -> {b.get('logicalType')}")
        if a.get("required") and not b.get("required"):
            br.append(f"{n}: no longer required (consumers may now see NULLs)")
        if not a.get("required") and b.get("required"):
            nb.append(f"{n}: now required")
        if a.get("primaryKey") and not b.get("primaryKey"):
            br.append(f"{n}: no longer part of the primary key")
        ha, hb = cprops(a).get("exprHash"), cprops(b).get("exprHash")
        if ha and hb and ha != hb:
            br.append(f"{n}: definition (formula) changed")
        if cprops(a).get("kind") != cprops(b).get("kind"):
            br.append(f"{n}: kind changed")
    orules = {q["name"]: q for q in oo.get("quality", []) or [] if q.get("name")}
    nrules = {q["name"]: q for q in no.get("quality", []) or [] if q.get("name")}
    for r in orules:
        if r not in nrules:
            br.append(f"rule removed: {r}")
        else:
            a = (orules[r].get("implementation") or {}).get("action")
            b = (nrules[r].get("implementation") or {}).get("action")
            if a and b and STRICTNESS.get(b, 0) < STRICTNESS.get(a, 0):
                br.append(f"rule {r} loosened: {a} -> {b}")
            elif a and b and a != b:
                nb.append(f"rule {r} tightened: {a} -> {b}")
    for r in nrules:
        if r not in orules:
            nb.append(f"rule added: {r}")
    sla = lambda c: {s["property"]: s.get("value") for s in c.get("slaProperties", []) or []}
    for k, v in sla(old).items():
        nv = sla(new).get(k)
        if k == "freshness" and nv is not None and v is not None and float(nv) > float(v):
            br.append(f"freshness SLA loosened: {v} -> {nv}")
        elif nv is None:
            br.append(f"SLA removed: {k}")
    if old.get("status") != "deprecated" and new.get("status") == "deprecated":
        nb.append("contract deprecated")

    def ver(c): return tuple(int(x) for x in str(c["version"]).split("."))
    changed = bool(br or nb) or checksum(old) != checksum(new)
    if changed and ver(new) <= ver(old):
        pol.append(f"contract changed but version not bumped ({old['version']} -> {new['version']})")
    if br and ver(new)[0] <= ver(old)[0]:
        pol.append(f"breaking changes require a MAJOR version bump ({old['version']} -> {new['version']})")
    return {"breaking": br, "non_breaking": nb, "policy_errors": pol}


In [ ]:
"""Run contracts against live Databricks objects: schema conformance (tables), semantic interface + tests + SLA (metric views).

Everything returns rows for governance.contract_results:
  (run_id, checked_at, contract_id, contract_version, layer, check_type, check_name, severity, passed, detail)
severity 'error' + passed=False means the contract is broken. 'warn' is recorded but never stops the job.
"""
from datetime import datetime

from pyspark.sql import SparkSession


CONTRACT_RESULTS_DDL = ("run_id STRING, checked_at TIMESTAMP, contract_id STRING, contract_version STRING, layer STRING, "
                        "check_type STRING, check_name STRING, severity STRING, passed BOOLEAN, detail STRING")
REGISTRY_DDL = ("contract_id STRING, name STRING, version STRING, status STRING, layer STRING, dataset STRING, owner STRING, "
                "checksum STRING, loaded_at TIMESTAMP, contract_json STRING")

SPARK_TYPES = {"string": {"string"}, "integer": {"int", "bigint", "smallint", "tinyint"}, "number": {"double", "float"},
               "boolean": {"boolean"}, "date": {"date"}, "timestamp": {"timestamp", "timestamp_ntz"}}


class ContractFailure(Exception):
    """A contract with severity=error was broken."""


def type_ok(logical: str, spark_type: str) -> bool:
    t = spark_type.lower()
    return t in SPARK_TYPES.get(logical, set()) or (logical == "number" and t.startswith("decimal"))


def _row(run_id, c, check_type, name, severity, passed, detail=""):
    return (run_id, datetime.now(), c["id"], str(c["version"]), layer_of(c), check_type, name, severity, bool(passed), str(detail)[:900])


def write_results(spark: SparkSession, rows: list):
    if rows:
        spark.createDataFrame(rows, CONTRACT_RESULTS_DDL).write.mode("append").saveAsTable("governance.contract_results")


def broken(rows: list) -> list:
    return [r for r in rows if r[7] == "error" and not r[8]]


def raise_if_broken(rows: list):
    bad = broken(rows)
    if bad:
        raise ContractFailure("Contract broken: " + "; ".join(f"{r[2]} {r[6]} ({r[9]})" for r in bad[:8]))


# --------------------------------------------------------------------------- tables (source / silver / gold)
def schema_conformance(spark: SparkSession, run_id: str, c: dict) -> list:
    obj = main_object(c)
    table = obj["physicalName"]
    rows = []
    try:
        df = spark.table(table)
    except Exception as e:  # noqa: BLE001
        return [_row(run_id, c, "schema", f"{table} exists", "error", False, str(e)[:300])]
    actual = {f.name.lower(): f.dataType.simpleString() for f in df.schema.fields}
    declared = [p["name"] for p in obj["properties"]]
    missing = [n for n in declared if n.lower() not in actual]
    rows.append(_row(run_id, c, "schema", "declared columns exist", "error", not missing,
                     f"missing: {', '.join(missing)}" if missing else f"{len(declared)} columns present"))
    wrong = [f"{p['name']} is {actual[p['name'].lower()]}, contract says {p['logicalType']}" for p in obj["properties"]
             if p["name"].lower() in actual and not type_ok(p["logicalType"], actual[p["name"].lower()])]
    rows.append(_row(run_id, c, "schema", "column types match", "error", not wrong, "; ".join(wrong) or "types match"))
    extra = [n for n in actual if not n.startswith("_") and n not in {d.lower() for d in declared}]
    rows.append(_row(run_id, c, "schema", "no undeclared columns", "warn", not extra,
                     f"undeclared: {', '.join(extra)}" if extra else "none"))
    for q in obj.get("quality", []) or []:
        if q.get("type") == "library" and q.get("metric") == "rowCount":
            n = df.count()
            floor = q.get("mustBeGreaterThan", 0)
            rows.append(_row(run_id, c, "schema", f"rowCount > {floor}", q.get("severity", "error"), n > floor, f"{n} rows"))
    return rows


# --------------------------------------------------------------------------- semantic views
def view_sql(view_fqn: str, dims=(), measures=(), where=None, limit=None) -> str:
    sel = [f"`{d}`" for d in dims] + [f"MEASURE(`{m}`) AS `{m}`" for m in measures]
    sql = f"SELECT {', '.join(sel)} FROM {view_fqn}"
    if where:
        sql += f" WHERE {where}"
    if dims:
        sql += " GROUP BY " + ", ".join(f"`{d}`" for d in dims)
    if limit:
        sql += f" LIMIT {int(limit)}"
    return sql


def make_executor(spark: SparkSession, view_fqn: str):
    """executor(request) -> DataFrame. request = {dims, measures, where?, limit?}. Tests can swap this out."""
    return lambda req: spark.sql(view_sql(view_fqn, req.get("dims", []), req.get("measures", []), req.get("where"), req.get("limit")))


def _num(x):
    return None if x is None else float(x)


def _close(a, b, tol):
    if a is None or b is None:
        return a is None and b is None
    return abs(float(a) - float(b)) <= tol


def semantic_checks(spark: SparkSession, run_id: str, c: dict, executor, now_sql_ok=True) -> list:
    obj = main_object(c)
    cp = cprops(c)
    strict = bool(cp.get("strictTypes"))
    props = obj["properties"]
    fields = [p for p in props if cprops(p).get("kind") == "field"]
    measures = [p for p in props if cprops(p).get("kind") == "measure"]
    first_measure = measures[0]["name"] if measures else None
    rows = []

    # 1) interface: every promised field and measure can be queried, with the promised type
    for p in props:
        kind = cprops(p).get("kind")
        req = {"dims": [p["name"]], "measures": [first_measure], "limit": 1} if kind == "field" else {"dims": [], "measures": [p["name"]]}
        try:
            df = executor(req)
            col = {f.name: f.dataType.simpleString() for f in df.schema.fields}[p["name"]]
            df.limit(1).collect()
            ok = type_ok(p["logicalType"], col)
            rows.append(_row(run_id, c, "interface", f"{kind} {p['name']} is queryable", "error", True))
            rows.append(_row(run_id, c, "interface", f"{kind} {p['name']} type is {p['logicalType']}",
                             "error" if strict else "warn", ok, f"actual {col}"))
        except Exception as e:  # noqa: BLE001
            rows.append(_row(run_id, c, "interface", f"{kind} {p['name']} is queryable", "error", False, str(e)[:300]))

    # 2) behaviour tests
    for q in obj.get("quality", []) or []:
        if q.get("engine") != ENGINE_SEM:
            continue
        i, name = q["implementation"], q["name"]
        sev = "warn" if q.get("severity") == "warning" else "error"      # warning-level tests are recorded, never block
        tol = float(i.get("tolerance", 0) or 0)
        dims = i.get("dimensions", []) or []
        try:
            if i["test_type"] == "reconcile":
                a = executor({"dims": dims, "measures": i["measures"]}).collect()
                b = spark.sql(i["compare_sql"]).collect()
                ka = {tuple(r[d] for d in dims): r for r in a}
                kb = {tuple(r[d] for d in dims): r for r in b}
                bad = [f"{k} missing on one side" for k in ka.keys() ^ kb.keys()]
                for k in ka.keys() & kb.keys():
                    for m in i["measures"]:
                        if not _close(ka[k][m], kb[k][m], tol):
                            bad.append(f"{k} {m}: view={ka[k][m]} gold={kb[k][m]}")
                rows.append(_row(run_id, c, "test", f"{name} {q.get('description', '')[:70]}", sev, not bad,
                                 f"{len(ka)} groups compared" if not bad else f"{len(bad)} mismatches, e.g. {bad[0]}"))
            elif i["test_type"] == "expect":
                got = executor({"dims": dims, "measures": i["measures"], "where": i.get("where")}).collect()
                index = {tuple(r[d] for d in dims): r for r in got}
                bad = []
                for e in i["expect"]:
                    k = tuple(e[d] for d in dims)
                    if k not in index:
                        bad.append(f"{k} not returned")
                        continue
                    for m in i["measures"]:
                        if not _close(index[k][m], e[m], tol):
                            bad.append(f"{k} {m}: got {index[k][m]}, expected {e[m]}")
                rows.append(_row(run_id, c, "test", f"{name} {q.get('description', '')[:70]}", sev, not bad,
                                 f"{len(i['expect'])} expected rows matched" if not bad else "; ".join(bad[:3])))
            elif i["test_type"] == "bounds":
                got = executor({"dims": dims, "measures": list(i["bounds"])}).collect()
                bad = [f"{[r[d] for d in dims]} {m}={r[m]}" for r in got for m, (lo, hi) in i["bounds"].items()
                       if r[m] is not None and not (lo <= float(r[m]) <= hi)]
                rows.append(_row(run_id, c, "test", f"{name} {q.get('description', '')[:70]}", sev, not bad,
                                 f"{len(got)} groups within bounds" if not bad else f"{len(bad)} out of bounds, e.g. {bad[0]}"))
        except Exception as e:  # noqa: BLE001
            rows.append(_row(run_id, c, "test", f"{name} ran", sev, False, str(e)[:300]))

    # 3) SLA: freshness of the gold tables this view depends on, and that their last gate was clean
    for s in c.get("slaProperties", []) or []:
        if s.get("property") != "freshness":
            continue
        hours = float(s["value"])
        for dep in cprops(obj).get("dependsOn", []) or []:
            table = dep.split(".")[-1]
            table = table[4:] if table.startswith("fbl_") else table
            try:
                n = spark.sql(f"SELECT COUNT(*) FROM governance.dq_table_stats WHERE layer = 'gold' AND table_name = '{table}' "
                              f"AND checked_at >= current_timestamp() - INTERVAL {int(hours)} HOURS").first()[0]
                rows.append(_row(run_id, c, "sla", f"{dep} refreshed within {int(hours)}h", "error", n > 0,
                                 "recent build found" if n else "no gold build recorded in the window"))
                bad = spark.sql(f"""SELECT COUNT(*) FROM governance.dq_rule_results r
                    WHERE r.layer = 'gold' AND r.table_name = '{table}' AND r.action = 'fail' AND r.violations > 0
                      AND r.run_id = (SELECT run_id FROM governance.dq_table_stats WHERE layer = 'gold' AND table_name = '{table}'
                                      ORDER BY checked_at DESC LIMIT 1)""").first()[0]
                rows.append(_row(run_id, c, "sla", f"{dep} last quality gate was clean", "error", bad == 0,
                                 "clean" if bad == 0 else f"{bad} failing rules in the latest build"))
            except Exception as e:  # noqa: BLE001
                rows.append(_row(run_id, c, "sla", f"{dep} freshness", "error", False, str(e)[:300]))
    return rows


In [ ]:
"""Fictional Baseball League (FBL): contracts -> bronze -> silver (+DQ) -> gold (+assertions) -> semantic -> report.

CONTRACT-DRIVEN: stage_contracts publishes data contracts into governance.data_contracts and compiles their quality rules
into governance.dq_rules. Every later stage reads the REGISTRY: expected columns (bronze), the typed SELECT (silver),
the schema promises (all layers) and the semantic checks (metric views). If no contracts are published the stages fall
back to the built-in definitions below, so the pipeline still runs.

Runs in the CURRENT catalog (run `USE CATALOG <catalog>` first) using two-part names: bronze.x, silver.x, gold.x, governance.x.
Stage functions are plain Python so the same code runs in a Databricks notebook and in the local test.
"""
import html as _html
import json
import os
from datetime import datetime
from functools import reduce

from pyspark.sql import DataFrame, functions as F


ENTITIES = ["team", "player", "batting"]          # order matters: parents before children (referential rules)
EXPECTED_COLUMNS = {
    "team": ["team_id", "team_name", "city", "league", "founded_year", "home_park", "payroll_cap_musd", "mascot"],
    "player": ["player_id", "first_name", "last_name", "birth_date", "bats", "throws", "team_id", "position",
               "height_in", "weight_lb", "nickname"],
    "batting": ["player_id", "season", "team_id", "g", "ab", "h", "doubles", "triples", "hr", "rbi", "bb", "so", "sb"],
}
FOLDERS = {"team": "teams", "player": "players", "batting": "batting"}

# ---- silver typing: trim, blank -> NULL, standardize case, try_cast (bad value -> NULL, caught later by not_null rules)
TYPED_SQL = {
"team": """
SELECT upper(trim(team_id)) AS team_id, NULLIF(trim(team_name), '') AS team_name, NULLIF(trim(city), '') AS city,
       initcap(NULLIF(trim(league), '')) AS league,
       try_cast(NULLIF(trim(founded_year), '') AS INT) AS founded_year, NULLIF(trim(home_park), '') AS home_park,
       try_cast(NULLIF(trim(payroll_cap_musd), '') AS DOUBLE) AS payroll_cap_musd, NULLIF(trim(mascot), '') AS mascot,
       _row_id, to_json(struct({RAW})) AS _raw, _ingested_at
FROM bronze.fbl_team t""",
"player": """
SELECT upper(trim(player_id)) AS player_id, NULLIF(trim(first_name), '') AS first_name, NULLIF(trim(last_name), '') AS last_name,
       try_cast(NULLIF(trim(birth_date), '') AS DATE) AS birth_date,
       upper(NULLIF(trim(bats), '')) AS bats, upper(NULLIF(trim(throws), '')) AS throws,
       upper(NULLIF(trim(team_id), '')) AS team_id, upper(NULLIF(trim(position), '')) AS position,
       try_cast(NULLIF(trim(height_in), '') AS INT) AS height_in, try_cast(NULLIF(trim(weight_lb), '') AS INT) AS weight_lb,
       NULLIF(trim(nickname), '') AS nickname,
       _row_id, to_json(struct({RAW})) AS _raw, _ingested_at
FROM bronze.fbl_player t""",
"batting": """
SELECT upper(trim(player_id)) AS player_id, try_cast(NULLIF(trim(season), '') AS INT) AS season, upper(trim(team_id)) AS team_id,
       try_cast(NULLIF(trim(g), '') AS INT) AS g, try_cast(NULLIF(trim(ab), '') AS INT) AS ab,
       try_cast(NULLIF(trim(h), '') AS INT) AS h, try_cast(NULLIF(trim(doubles), '') AS INT) AS doubles,
       try_cast(NULLIF(trim(triples), '') AS INT) AS triples, try_cast(NULLIF(trim(hr), '') AS INT) AS hr,
       try_cast(NULLIF(trim(rbi), '') AS INT) AS rbi, try_cast(NULLIF(trim(bb), '') AS INT) AS bb,
       try_cast(NULLIF(trim(so), '') AS INT) AS so, try_cast(NULLIF(trim(sb), '') AS INT) AS sb,
       _row_id, to_json(struct({RAW})) AS _raw, _ingested_at
FROM bronze.fbl_batting t""",
}

# ---- gold: optional descriptive text gets an explicit label (never default a numeric measure)
GOLD_SQL = {
"dim_team": """
SELECT team_id, team_name, city, league, founded_year, home_park, payroll_cap_musd, coalesce(mascot, 'None') AS mascot
FROM silver.fbl_team""",
"dim_player": """
SELECT player_id, first_name, last_name, concat(first_name, ' ', last_name) AS full_name, birth_date, bats, throws,
       team_id, position, height_in, weight_lb, coalesce(nickname, 'None') AS nickname
FROM silver.fbl_player""",
"fact_batting": """
SELECT player_id, season, team_id, g, ab, h, doubles, triples, hr, rbi, bb, so, sb,
       h - doubles - triples - hr AS singles,
       h + doubles + 2 * triples + 3 * hr AS total_bases,
       ab + bb AS plate_appearances_approx
FROM silver.fbl_batting""",
}
GOLD_SOURCE = {"dim_team": "team", "dim_player": "player", "fact_batting": "batting"}


def typed_sql(entity: str) -> str:
    """Typed silver SELECT. _raw keeps the ORIGINAL bronze text of the row so a rejected record shows what was really sent."""
    raw_cols = ", ".join(f"t.`{c}`" for c in EXPECTED_COLUMNS[entity])
    return TYPED_SQL[entity].replace("{RAW}", raw_cols)


def _append(df: DataFrame, table: str):
    df.write.mode("append").saveAsTable(table)


# --------------------------------------------------------------------------- contract registry
def load_registry(spark):
    """Published, active contracts as a ContractSet, or None if nothing has been published yet."""
    try:
        rows = spark.table("governance.data_contracts").where("status = 'active'").collect()
    except Exception:  # noqa: BLE001  (table missing before setup)
        return None
    return ContractSet([json.loads(r["contract_json"]) for r in rows]) if rows else None


def _entity_contracts(cs, layer, entity):
    return cs.entity(layer, entity, "fbl") if cs else None


# --------------------------------------------------------------------------- setup
def stage_setup(spark, rules_csv_text: str = None, reload_rules: bool = False):
    """Create schemas, governance tables and views. rules_csv_text is only for the legacy CSV rules mode."""
    for s in ["bronze", "silver", "gold", "semantic", "governance"]:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {s}")
    spark.sql(f"CREATE TABLE IF NOT EXISTS governance.dq_rules ({RULES_DDL})")
    spark.sql(f"CREATE TABLE IF NOT EXISTS governance.dq_rule_results ({RESULTS_DDL})")
    spark.sql(f"CREATE TABLE IF NOT EXISTS governance.dq_table_stats ({STATS_DDL})")
    spark.sql(f"CREATE TABLE IF NOT EXISTS governance.dq_quarantine ({QUARANTINE_DDL})")
    spark.sql(f"CREATE TABLE IF NOT EXISTS governance.data_contracts ({REGISTRY_DDL})")
    spark.sql(f"CREATE TABLE IF NOT EXISTS governance.data_contracts_history ({REGISTRY_DDL})")
    spark.sql(f"CREATE TABLE IF NOT EXISTS governance.contract_results ({CONTRACT_RESULTS_DDL})")
    if rules_csv_text and (reload_rules or spark.table("governance.dq_rules").count() == 0):
        rules = parse_rules_csv(rules_csv_text)
        cols = [c.split()[0] for c in RULES_DDL.split(", ")]
        spark.createDataFrame([tuple(r[c] for c in cols) for r in rules], RULES_DDL).write.mode("overwrite").saveAsTable("governance.dq_rules")
    spark.sql("""CREATE OR REPLACE VIEW governance.v_dq_summary AS
        SELECT * FROM governance.dq_table_stats
        WHERE run_id = (SELECT run_id FROM governance.dq_table_stats ORDER BY checked_at DESC LIMIT 1)""")
    spark.sql("""CREATE OR REPLACE VIEW governance.v_dq_open_rejects AS
        SELECT run_id, source_table, row_id, rule_ids, payload, status, fix_notes
        FROM governance.dq_quarantine WHERE disposition = 'rejected' AND status = 'open'""")
    spark.sql("""CREATE OR REPLACE VIEW governance.v_contract_status AS
        SELECT contract_id, name, version, status, layer, dataset, owner, loaded_at FROM governance.data_contracts""")
    spark.sql("""CREATE OR REPLACE VIEW governance.v_contract_failures AS
        SELECT * FROM governance.contract_results WHERE passed = FALSE
        AND run_id = (SELECT run_id FROM governance.contract_results ORDER BY checked_at DESC LIMIT 1)""")
    print("setup complete; active rules:", spark.table("governance.dq_rules").where("active = 'Y'").count())


# --------------------------------------------------------------------------- publish contracts
def stage_contracts(spark, run_id: str, contracts: list, definitions: dict = None, origin: str = "unknown"):
    """Validate (lint) and publish contracts: registry + history + compiled DQ rules. Fails the run if lint finds errors."""
    cs = ContractSet(contracts)
    parsed = None
    if definitions:
        try:
            parsed = {name: load_metric_view_definition(sql) for name, sql in definitions.items()}
        except ImportError as e:
            print("NOTE: skipping metric-view definition checks (PyYAML missing); they are enforced in CI.", str(e)[:80])
    findings = lint(cs, parsed)
    rows = [(run_id, datetime.now(), cid, "", "", "lint", msg[:200], sev, False, "") for sev, cid, msg in findings]
    errors = [f for f in findings if f[0] == "error"]
    for sev, cid, msg in findings:
        print(f"  lint {sev}: {cid}: {msg}")
    if errors:
        write_results(spark, [(r[0], r[1], r[2], "0", "contract", r[5], r[6], r[7], False, "lint") for r in rows])
        raise ContractFailure(f"{len(errors)} contract lint error(s): " + "; ".join(f"{c}: {m}" for _, c, m in errors[:5]))

    now = datetime.now()
    reg = [(c["id"], c.get("name"), str(c["version"]), c["status"], layer_of(c), main_object(c)["physicalName"],
            cprops(c).get("owner"), checksum(c), now, json.dumps(c, sort_keys=True)) for c in contracts]
    spark.createDataFrame(reg, REGISTRY_DDL).write.mode("overwrite").saveAsTable("governance.data_contracts")
    hist = spark.table("governance.data_contracts_history")
    new = spark.createDataFrame(reg, REGISTRY_DDL).join(hist.select("contract_id", "checksum"), ["contract_id", "checksum"], "left_anti")
    n_new = new.count()
    if n_new:
        new.write.mode("append").saveAsTable("governance.data_contracts_history")
    rules = compile_all_rules(cs)
    cols = [c.split()[0] for c in RULES_DDL.split(", ")]
    spark.createDataFrame([tuple(r[c] for c in cols) for r in rules], RULES_DDL).write.mode("overwrite").saveAsTable("governance.dq_rules")
    warns = [(r[0], r[1], r[2], "0", "contract", r[5], r[6], r[7], False, "lint") for r in rows]
    write_results(spark, warns + [(run_id, now, "(all)", "", "contract", "publish", f"published {len(contracts)} contracts from {origin}",
                                    "info", True, f"{n_new} new or changed versions; {len(rules)} rules compiled")])
    print(f"published {len(contracts)} contracts ({n_new} new/changed) from {origin}; compiled {len(rules)} rules")


# --------------------------------------------------------------------------- bronze
def stage_bronze(spark, landing_dir: str, run_id: str):
    cs = load_registry(spark)
    results, contract_rows, now = [], [], datetime.now()
    for e in ENTITIES:
        sc = _entity_contracts(cs, "source", e)
        folder = cprops(main_object(sc))["landingFolder"] if sc else FOLDERS[e]
        expected = expected_columns(sc) if sc else EXPECTED_COLUMNS[e]
        path = f"{landing_dir}/{folder}/"
        df = spark.read.option("header", "true").option("inferSchema", "false").csv(path)   # all strings
        missing = [c for c in expected if c not in df.columns]
        n = df.count()
        results.append((run_id, now, "bronze", e, "BRZ-001", "columns_present", "fail", len(missing), n,
                        "Expected columns are present" + (f" (missing: {', '.join(missing)})" if missing else "")))
        results.append((run_id, now, "bronze", e, "BRZ-002", "not_empty", "fail", 1 if n == 0 else 0, n, "Source file is not empty"))
        if not missing:
            (df.select(*[F.col(c) for c in df.columns],
                       F.monotonically_increasing_id().alias("_row_id"),
                       F.col("_metadata.file_path").alias("_source_file"),
                       F.current_timestamp().alias("_ingested_at"))
               .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"bronze.fbl_{e}"))
            if sc:
                contract_rows += schema_conformance(spark, run_id, sc)
        print(f"bronze.fbl_{e}: {n} rows" + (f"  [contract {sc['id']} v{sc['version']}]" if sc else "  [no contract: built-in columns]"))
    res = spark.createDataFrame(results, RESULTS_DDL)
    spark.createDataFrame(results, RESULTS_DDL).write.mode("append").saveAsTable("governance.dq_rule_results")
    write_results(spark, contract_rows)
    raise_if_failed(res)
    raise_if_broken(contract_rows)


# --------------------------------------------------------------------------- silver
def stage_silver(spark, run_id: str):
    cs = load_registry(spark)
    all_results, contract_rows = [], []
    for e in ENTITIES:
        sv, src = _entity_contracts(cs, "silver", e), _entity_contracts(cs, "source", e)
        sql = typed_select(sv, src) if sv and src else typed_sql(e)
        typed = spark.sql(sql)
        rules = load_rules(spark, "silver", e)
        clean, log_df, res, stats = apply_rules(spark, typed, rules, run_id, "silver", e, mode="filter")
        clean.drop("_raw").write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"silver.fbl_{e}")
        _append(log_df, "governance.dq_quarantine")
        _append(res, "governance.dq_rule_results")
        _append(_stats_df(spark, stats), "governance.dq_table_stats")
        all_results.append(res)
        if sv:
            contract_rows += schema_conformance(spark, run_id, sv)
        print(f"silver.fbl_{e}: in={stats['rows_in']} deduped={stats['rows_deduped']} rejected={stats['rows_rejected']} "
              f"out={stats['rows_out']}  [{'contract ' + sv['id'] + ' v' + sv['version'] if sv else 'built-in typing'}]")
    write_results(spark, contract_rows)
    raise_if_failed(reduce(DataFrame.unionByName, all_results))
    raise_if_broken(contract_rows)


def _stats_df(spark, s: dict) -> DataFrame:
    return (spark.createDataFrame([(s["run_id"], s["layer"], s["table_name"], s["rows_in"], s["rows_deduped"],
                                    s["rows_standardized"], s["rows_warned"], s["rows_rejected"], s["rows_out"])],
                                  "run_id STRING, layer STRING, table_name STRING, rows_in BIGINT, rows_deduped BIGINT, "
                                  "rows_standardized BIGINT, rows_warned BIGINT, rows_rejected BIGINT, rows_out BIGINT")
            .withColumn("checked_at", F.current_timestamp())
            .select(*[c.split()[0] for c in STATS_DDL.split(", ")]))


# --------------------------------------------------------------------------- gold
def stage_gold(spark, run_id: str):
    cs = load_registry(spark)
    all_results, contract_rows = [], []
    for g, src in GOLD_SOURCE.items():
        gc = _entity_contracts(cs, "gold", g)
        df = spark.sql(GOLD_SQL[g])
        df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"gold.fbl_{g}")
        built = spark.table(f"gold.fbl_{g}")
        rules = load_rules(spark, "gold", g)
        _, _, res, stats = apply_rules(spark, built, rules, run_id, "gold", g, mode="assert")
        # reconciliation: gold must neither lose nor invent rows relative to clean silver
        recon_table = cprops(main_object(gc)).get("reconcilesTo") if gc else f"silver.fbl_{src}"
        silver_n, gold_n = spark.table(recon_table).count(), built.count()
        rec = spark.createDataFrame([(run_id, datetime.now(), "gold", g, "GLD-RECON", "reconcile", "fail",
                                      abs(silver_n - gold_n), gold_n, f"Row count gold = silver ({silver_n} vs {gold_n})")], RESULTS_DDL)
        res = res.unionByName(rec)
        _append(res, "governance.dq_rule_results")
        _append(_stats_df(spark, stats), "governance.dq_table_stats")
        all_results.append(res)
        if gc:
            contract_rows += schema_conformance(spark, run_id, gc)
        print(f"gold.fbl_{g}: {gold_n} rows  [{'contract ' + gc['id'] + ' v' + gc['version'] if gc else 'no contract'}]")
    write_results(spark, contract_rows)
    raise_if_failed(reduce(DataFrame.unionByName, all_results))
    raise_if_broken(contract_rows)


# --------------------------------------------------------------------------- semantic (metric views + their contracts)
def stage_semantic(spark, run_id: str, catalog: str, definitions: dict = None, build: bool = True,
                   executor_factory=None, product: str = "fbl"):
    """Create the metric views from their definitions, then check each view against its semantic contract."""
    cs = load_registry(spark)
    if not cs:
        raise ContractFailure("no contracts published: run the contracts stage first")
    rows = []
    for c in cs.layer("semantic", product):
        obj = main_object(c)
        defname = os.path.basename(cprops(obj)["definition"])
        view_fqn = f"{catalog}.{obj['physicalName']}"
        if build:
            sql = (definitions or {}).get(defname)
            if not sql:
                raise ContractFailure(f"{c['id']}: definition {defname} not found (upload it with the contracts bundle)")
            spark.sql(render(sql, catalog).rstrip().rstrip(";"))
            print(f"created {view_fqn}")
        executor = executor_factory(c, view_fqn) if executor_factory else make_executor(spark, view_fqn)
        got = semantic_checks(spark, run_id, c, executor)
        rows += got
        bad = [r for r in got if r[7] == "error" and not r[8]]
        print(f"{c['id']} v{c['version']}: {len(got)} checks, {len(bad)} broken")
    write_results(spark, rows)
    raise_if_broken(rows)


# --------------------------------------------------------------------------- report
def build_report(spark, run_id: str) -> dict:
    stats = spark.table("governance.dq_table_stats").where(F.col("run_id") == run_id).orderBy("checked_at").collect()
    viol = (spark.table("governance.dq_rule_results").where((F.col("run_id") == run_id) & (F.col("violations") > 0))
            .orderBy("layer", "table_name", "rule_id").collect())
    rej = (spark.table("governance.dq_quarantine").where((F.col("run_id") == run_id) & (F.col("disposition") == "rejected"))
           .orderBy("source_table", "row_id").collect())
    fixed = (spark.table("governance.dq_quarantine").where((F.col("run_id") == run_id) & (F.col("disposition") == "auto_corrected"))
             .groupBy("source_table", "rule_ids").count().orderBy("source_table", "rule_ids").collect())
    cres = [r for r in spark.table("governance.contract_results").where((F.col("run_id") == run_id) & (F.col("passed") == False)).collect()]  # noqa: E712
    n_cbroken = sum(1 for r in cres if r["severity"] == "error")
    n_rej = len(rej)
    n_failed = sum(1 for v in viol if v["action"] == "fail")
    lines = [f"DQ run {run_id}: {n_rej} rows rejected, {sum(f['count'] for f in fixed)} auto-corrections, {n_failed} failed rules, "
             f"{n_cbroken} broken contract checks"]
    for r in cres[:30]:
        lines.append(f"  CONTRACT {r['severity'].upper()} {r['contract_id']} [{r['check_type']}] {r['check_name']}: {r['detail'][:120]}")
    for s in stats:
        lines.append(f"  {s['layer']}.{s['table_name']}: in={s['rows_in']} deduped={s['rows_deduped']} "
                     f"rejected={s['rows_rejected']} out={s['rows_out']}")
    for r in rej[:50]:
        lines.append(f"  REJECT {r['source_table']} row {r['row_id']} [{r['rule_ids']}] {r['payload'][:140]}")
    esc = _html.escape
    def table(head, rows):
        return ("<table border='1' cellpadding='4' style='border-collapse:collapse;font-family:sans-serif;font-size:13px'><tr>"
                + "".join(f"<th>{esc(h)}</th>" for h in head) + "</tr>"
                + "".join("<tr>" + "".join(f"<td>{esc(str(c))}</td>" for c in row) + "</tr>" for row in rows) + "</table>")
    html = (f"<h2>Data quality report</h2><p>Run <code>{esc(run_id)}</code>: <b>{n_rej}</b> rows rejected, "
            f"<b>{n_failed}</b> failed rules.</p><h3>Row flow by table</h3>"
            + table(["layer", "table", "in", "deduped", "standardized", "warned", "rejected", "out"],
                    [(s["layer"], s["table_name"], s["rows_in"], s["rows_deduped"], s["rows_standardized"], s["rows_warned"],
                      s["rows_rejected"], s["rows_out"]) for s in stats])
            + "<h3>Rules with violations</h3>"
            + table(["layer", "table", "rule", "action", "violations", "description"],
                    [(v["layer"], v["table_name"], v["rule_id"], v["action"], v["violations"], v["description"]) for v in viol])
            + "<h3>Auto-corrected (no action needed)</h3>"
            + table(["table", "rule", "rows"], [(f["source_table"], f["rule_ids"], f["count"]) for f in fixed])
            + "<h3>Contract checks that did not pass</h3>"
            + table(["contract", "type", "check", "severity", "detail"],
                    [(r["contract_id"], r["check_type"], r["check_name"], r["severity"], r["detail"]) for r in cres])
            + "<h3>Rejected rows: kick back to the source owner</h3>"
            + table(["table", "row", "rules", "original record"], [(r["source_table"], r["row_id"], r["rule_ids"], r["payload"]) for r in rej]))
    return {"text": "\n".join(lines), "html": html, "rejected": n_rej, "failed_rules": n_failed, "contract_failures": n_cbroken}


In [ ]:
EMBEDDED_BUNDLE_JSON = (
    '{"bundleVersion": 1, "contracts": [{"checksum": "89d4531cd5cd06da", "contract": {"apiVersion": "v3.1'
    '.0", "customProperties": [{"property": "layer", "value": "gold"}, {"property": "owner", "value": "Da'
    'ta Engineering"}, {"property": "dependsOn", "value": "fbl.silver.player"}, {"property": "consumers",'
    ' "value": ["fbl.semantic.mv_fbl_batting", "fbl.semantic.mv_fbl_roster"]}], "dataProduct": "fbl", "de'
    'scription": {"limitations": "Gold never contains NULLs. Optional text is labeled \'None\'. Numeric fac'
    'ts are never defaulted: bad rows are rejected in silver.", "purpose": "Player dimension: one row per'
    ' player.", "usage": "Safe for analysts and the semantic layer. Prefer the metric views for ratios."}'
    ', "domain": "baseball_operations", "id": "fbl.gold.dim_player", "kind": "DataContract", "name": "FBL'
    ' dim_player (gold)", "schema": [{"customProperties": [{"property": "entity", "value": "dim_player"},'
    ' {"property": "reconcilesTo", "value": "silver.fbl_player"}], "description": "Player dimension: one '
    'row per player.", "logicalType": "object", "name": "dim_player", "physicalName": "gold.fbl_dim_playe'
    'r", "physicalType": "table", "properties": [{"description": "Player identifier, e.g. P001. Primary k'
    'ey.", "logicalType": "string", "name": "player_id", "physicalType": "STRING", "primaryKey": true, "p'
    'rimaryKeyPosition": 1, "required": true}, {"description": "Given name.", "logicalType": "string", "n'
    'ame": "first_name", "physicalType": "STRING", "required": true}, {"description": "Family name.", "lo'
    'gicalType": "string", "name": "last_name", "physicalType": "STRING", "required": true}, {"descriptio'
    'n": "first_name + last_name", "logicalType": "string", "name": "full_name", "physicalType": "STRING"'
    ', "required": true}, {"classification": "personal", "description": "Date of birth. Personal data. Pl'
    'ayer must be at least 16.", "logicalType": "date", "name": "birth_date", "physicalType": "DATE", "re'
    'quired": true}, {"description": "Batting side: L, R or B (switch).", "logicalType": "string", "name"'
    ': "bats", "physicalType": "STRING", "required": true}, {"description": "Throwing hand: L or R.", "lo'
    'gicalType": "string", "name": "throws", "physicalType": "STRING", "required": true}, {"description":'
    ' "Team the player belongs to.", "logicalType": "string", "name": "team_id", "physicalType": "STRING"'
    ', "relationships": [{"to": "dim_team.team_id", "type": "foreignKey"}], "required": true}, {"descript'
    'ion": "Position code: P, C, 1B, 2B, 3B, SS, LF, CF, RF or DH.", "logicalType": "string", "name": "po'
    'sition", "physicalType": "STRING", "required": true}, {"description": "Height in inches (60 to 84)."'
    ', "logicalType": "integer", "name": "height_in", "physicalType": "INT", "required": true}, {"descrip'
    'tion": "Weight in pounds (120 to 350 expected).", "logicalType": "integer", "name": "weight_lb", "ph'
    'ysicalType": "INT", "required": true}, {"description": "Gold guarantee: never NULL; the label \'None\''
    ' means not provided.", "logicalType": "string", "name": "nickname", "physicalType": "STRING", "requi'
    'red": true}], "quality": [{"description": "Gold table must not be empty.", "dimension": "completenes'
    's", "metric": "rowCount", "mustBeGreaterThan": 0, "severity": "error", "type": "library"}, {"descrip'
    'tion": "Gold guarantee: no NULL in any dim_player column", "dimension": "completeness", "engine": "f'
    'bl-dq-engine", "implementation": {"action": "fail", "columns": "*", "expression": "", "owner": "Data'
    ' Engineering", "rule_type": "no_nulls"}, "name": "GLD-002", "severity": "error", "type": "custom"}, '
    '{"description": "dim_player grain: one row per player_id", "dimension": "uniqueness", "engine": "fbl'
    '-dq-engine", "implementation": {"action": "fail", "columns": ["player_id"], "expression": "", "owner'
    '": "Data Engineering", "rule_type": "unique"}, "name": "GLD-005", "severity": "error", "type": "cust'
    'om"}, {"description": "Every player must belong to a team in dim_team", "dimension": "consistency", '
    '"engine": "fbl-dq-engine", "implementation": {"action": "fail", "columns": ["team_id"], "expression"'
    ': "gold.fbl_dim_team.team_id", "owner": "Data Engineering", "rule_type": "referential"}, "name": "GL'
    'D-009", "severity": "error", "type": "custom"}]}], "slaProperties": [{"element": "dim_player", "prop'
    'erty": "freshness", "unit": "h", "value": 24}, {"element": "dim_player", "property": "frequency", "u'
    'nit": "d", "value": 1}], "status": "active", "tags": ["fbl", "gold"], "version": "1.0.0"}, "id": "fb'
    'l.gold.dim_player"}, {"checksum": "caa7b331508bb106", "contract": {"apiVersion": "v3.1.0", "customPr'
    'operties": [{"property": "layer", "value": "gold"}, {"property": "owner", "value": "Data Engineering'
    '"}, {"property": "dependsOn", "value": "fbl.silver.team"}, {"property": "consumers", "value": ["fbl.'
    'semantic.mv_fbl_batting", "fbl.semantic.mv_fbl_roster"]}], "dataProduct": "fbl", "description": {"li'
    'mitations": "Gold never contains NULLs. Optional text is labeled \'None\'. Numeric facts are never def'
    'aulted: bad rows are rejected in silver.", "purpose": "Team dimension: one row per team.", "usage": '
    '"Safe for analysts and the semantic layer. Prefer the metric views for ratios."}, "domain": "basebal'
    'l_operations", "id": "fbl.gold.dim_team", "kind": "DataContract", "name": "FBL dim_team (gold)", "sc'
    'hema": [{"customProperties": [{"property": "entity", "value": "dim_team"}, {"property": "reconcilesT'
    'o", "value": "silver.fbl_team"}], "description": "Team dimension: one row per team.", "logicalType":'
    ' "object", "name": "dim_team", "physicalName": "gold.fbl_dim_team", "physicalType": "table", "proper'
    'ties": [{"description": "Team identifier, e.g. T01. Primary key.", "logicalType": "string", "name": '
    '"team_id", "physicalType": "STRING", "primaryKey": true, "primaryKeyPosition": 1, "required": true},'
    ' {"description": "Full team name.", "logicalType": "string", "name": "team_name", "physicalType": "S'
    'TRING", "required": true}, {"description": "Home city.", "logicalType": "string", "name": "city", "p'
    'hysicalType": "STRING", "required": true}, {"description": "League: North or South.", "logicalType":'
    ' "string", "name": "league", "physicalType": "STRING", "required": true}, {"description": "Year the '
    'franchise was founded (league started in 2000).", "logicalType": "integer", "name": "founded_year", '
    '"physicalType": "INT", "required": true}, {"description": "Home ballpark.", "logicalType": "string",'
    ' "name": "home_park", "physicalType": "STRING", "required": true}, {"description": "Payroll cap in m'
    'illions of USD. Must be positive.", "logicalType": "number", "name": "payroll_cap_musd", "physicalTy'
    'pe": "DOUBLE", "required": true}, {"description": "Team mascot. Gold guarantee: never NULL; the labe'
    'l \'None\' means not provided.", "logicalType": "string", "name": "mascot", "physicalType": "STRING", '
    '"required": true}], "quality": [{"description": "Gold table must not be empty.", "dimension": "compl'
    'eteness", "metric": "rowCount", "mustBeGreaterThan": 0, "severity": "error", "type": "library"}, {"d'
    'escription": "Gold guarantee: no NULL in any dim_team column", "dimension": "completeness", "engine"'
    ': "fbl-dq-engine", "implementation": {"action": "fail", "columns": "*", "expression": "", "owner": "'
    'Data Engineering", "rule_type": "no_nulls"}, "name": "GLD-001", "severity": "error", "type": "custom'
    '"}, {"description": "dim_team grain: one row per team_id", "dimension": "uniqueness", "engine": "fbl'
    '-dq-engine", "implementation": {"action": "fail", "columns": ["team_id"], "expression": "", "owner":'
    ' "Data Engineering", "rule_type": "unique"}, "name": "GLD-004", "severity": "error", "type": "custom'
    '"}]}], "slaProperties": [{"element": "dim_team", "property": "freshness", "unit": "h", "value": 24},'
    ' {"element": "dim_team", "property": "frequency", "unit": "d", "value": 1}], "status": "active", "ta'
    'gs": ["fbl", "gold"], "version": "1.0.0"}, "id": "fbl.gold.dim_team"}, {"checksum": "6d515499b291e2d'
    '9", "contract": {"apiVersion": "v3.1.0", "customProperties": [{"property": "layer", "value": "gold"}'
    ', {"property": "owner", "value": "Data Engineering"}, {"property": "dependsOn", "value": "fbl.silver'
    '.batting"}, {"property": "consumers", "value": ["fbl.semantic.mv_fbl_batting", "fbl.semantic.mv_fbl_'
    'roster"]}], "dataProduct": "fbl", "description": {"limitations": "Gold never contains NULLs. Optiona'
    'l text is labeled \'None\'. Numeric facts are never defaulted: bad rows are rejected in silver.", "pur'
    'pose": "Batting fact: one row per player, season and team. Additive components only; ratios live in '
    'the semantic layer.", "usage": "Safe for analysts and the semantic layer. Prefer the metric views fo'
    'r ratios."}, "domain": "baseball_operations", "id": "fbl.gold.fact_batting", "kind": "DataContract",'
    ' "name": "FBL fact_batting (gold)", "schema": [{"customProperties": [{"property": "entity", "value":'
    ' "fact_batting"}, {"property": "reconcilesTo", "value": "silver.fbl_batting"}], "description": "Batt'
    'ing fact: one row per player, season and team. Additive components only; ratios live in the semantic'
    ' layer.", "logicalType": "object", "name": "fact_batting", "physicalName": "gold.fbl_fact_batting", '
    '"physicalType": "table", "properties": [{"description": "Player. Part of the key.", "logicalType": "'
    'string", "name": "player_id", "physicalType": "STRING", "primaryKey": true, "primaryKeyPosition": 1,'
    ' "relationships": [{"to": "dim_player.player_id", "type": "foreignKey"}], "required": true}, {"descr'
    'iption": "Season year. Part of the key.", "logicalType": "integer", "name": "season", "physicalType"'
    ': "INT", "primaryKey": true, "primaryKeyPosition": 2, "required": true}, {"description": "Team playe'
    'd for. Part of the key.", "logicalType": "string", "name": "team_id", "physicalType": "STRING", "pri'
    'maryKey": true, "primaryKeyPosition": 3, "relationships": [{"to": "dim_team.team_id", "type": "forei'
    'gnKey"}], "required": true}, {"description": "Games played (0 to 162).", "logicalType": "integer", "'
    'name": "g", "physicalType": "INT", "required": true}, {"description": "At bats.", "logicalType": "in'
    'teger", "name": "ab", "physicalType": "INT", "required": true}, {"description": "Hits. Cannot exceed'
    ' at bats.", "logicalType": "integer", "name": "h", "physicalType": "INT", "required": true}, {"descr'
    'iption": "Doubles.", "logicalType": "integer", "name": "doubles", "physicalType": "INT", "required":'
    ' true}, {"description": "Triples.", "logicalType": "integer", "name": "triples", "physicalType": "IN'
    'T", "required": true}, {"description": "Home runs. Cannot exceed hits.", "logicalType": "integer", "'
    'name": "hr", "physicalType": "INT", "required": true}, {"description": "Runs batted in.", "logicalTy'
    'pe": "integer", "name": "rbi", "physicalType": "INT", "required": true}, {"description": "Walks.", "'
    'logicalType": "integer", "name": "bb", "physicalType": "INT", "required": true}, {"description": "St'
    'rikeouts.", "logicalType": "integer", "name": "so", "physicalType": "INT", "required": true}, {"desc'
    'ription": "Stolen bases. Cannot be negative.", "logicalType": "integer", "name": "sb", "physicalType'
    '": "INT", "required": true}, {"description": "Derived: hits - doubles - triples - home runs.", "logi'
    'calType": "integer", "name": "singles", "physicalType": "INT", "required": true}, {"description": "D'
    'erived: 1B + 2*2B + 3*3B + 4*HR.", "logicalType": "integer", "name": "total_bases", "physicalType": '
    '"INT", "required": true}, {"description": "Derived: at bats + walks (approximation).", "logicalType"'
    ': "integer", "name": "plate_appearances_approx", "physicalType": "INT", "required": true}], "quality'
    '": [{"description": "Gold table must not be empty.", "dimension": "completeness", "metric": "rowCoun'
    't", "mustBeGreaterThan": 0, "severity": "error", "type": "library"}, {"description": "Gold guarantee'
    ': no NULL in any fact_batting column", "dimension": "completeness", "engine": "fbl-dq-engine", "impl'
    'ementation": {"action": "fail", "columns": "*", "expression": "", "owner": "Data Engineering", "rule'
    '_type": "no_nulls"}, "name": "GLD-003", "severity": "error", "type": "custom"}, {"description": "fac'
    't_batting grain: one row per player, season, team", "dimension": "uniqueness", "engine": "fbl-dq-eng'
    'ine", "implementation": {"action": "fail", "columns": ["player_id", "season", "team_id"], "expressio'
    'n": "", "owner": "Data Engineering", "rule_type": "unique"}, "name": "GLD-006", "severity": "error",'
    ' "type": "custom"}, {"description": "Every batting row must have a player in dim_player", "dimension'
    '": "consistency", "engine": "fbl-dq-engine", "implementation": {"action": "fail", "columns": ["playe'
    'r_id"], "expression": "gold.fbl_dim_player.player_id", "owner": "Data Engineering", "rule_type": "re'
    'ferential"}, "name": "GLD-007", "severity": "error", "type": "custom"}, {"description": "Every batti'
    'ng row must have a team in dim_team", "dimension": "consistency", "engine": "fbl-dq-engine", "implem'
    'entation": {"action": "fail", "columns": ["team_id"], "expression": "gold.fbl_dim_team.team_id", "ow'
    'ner": "Data Engineering", "rule_type": "referential"}, "name": "GLD-008", "severity": "error", "type'
    '": "custom"}, {"description": "Business rules re-checked on gold: hits, home runs and at bats are co'
    'nsistent", "dimension": "accuracy", "engine": "fbl-dq-engine", "implementation": {"action": "fail", '
    '"columns": ["h"], "expression": "h <= ab AND hr <= h AND ab <= g * 6", "owner": "Data Engineering", '
    '"rule_type": "custom"}, "name": "GLD-010", "severity": "error", "type": "custom"}]}], "slaProperties'
    '": [{"element": "fact_batting", "property": "freshness", "unit": "h", "value": 24}, {"element": "fac'
    't_batting", "property": "frequency", "unit": "d", "value": 1}], "status": "active", "tags": ["fbl", '
    '"gold"], "version": "1.0.0"}, "id": "fbl.gold.fact_batting"}, {"checksum": "210d6611c9554670", "cont'
    'ract": {"apiVersion": "v3.1.0", "customProperties": [{"property": "layer", "value": "semantic"}, {"p'
    'roperty": "owner", "value": "Analytics Engineering"}, {"property": "strictTypes", "value": false}], '
    '"dataProduct": "fbl", "description": {"limitations": "OBP and OPS are approximations (no hit-by-pitc'
    'h or sacrifice-fly data). Distinct counts (batters) cannot be summed across groups. Only rows that p'
    'assed silver quality are included.", "purpose": "Governed batting metrics for the Fictional Baseball'
    ' League. Consumers group and filter by season, player, team, league, city, position or batting hand.'
    '", "usage": "SELECT dim, MEASURE(measure) FROM semantic.mv_fbl_batting GROUP BY dim. Do not query th'
    'e gold tables directly for ratios."}, "domain": "baseball_operations", "id": "fbl.semantic.mv_fbl_ba'
    'tting", "kind": "DataContract", "name": "mv_fbl_batting (semantic)", "schema": [{"customProperties":'
    ' [{"property": "definition", "value": "sql/fbl/metric_views/mv_fbl_batting.sql"}, {"property": "depe'
    'ndsOn", "value": ["gold.fbl_fact_batting", "gold.fbl_dim_player", "gold.fbl_dim_team"]}, {"property"'
    ': "consumers", "value": ["bi_dashboards", "genie_space_fbl"]}], "description": "Governed batting met'
    'rics for the Fictional Baseball League. Consumers group and filter by season, player, team, league, '
    'city, position or batting hand.", "logicalType": "object", "name": "mv_fbl_batting", "physicalName":'
    ' "semantic.mv_fbl_batting", "physicalType": "view", "properties": [{"customProperties": [{"property"'
    ': "kind", "value": "field"}, {"property": "exprHash", "value": "18c81f0acf57"}], "description": "Sea'
    'son year.", "logicalType": "integer", "name": "season", "required": false}, {"customProperties": [{"'
    'property": "kind", "value": "field"}, {"property": "exprHash", "value": "d28d7eea452f"}], "descripti'
    'on": "Player full name.", "logicalType": "string", "name": "player_name", "required": false}, {"cust'
    'omProperties": [{"property": "kind", "value": "field"}, {"property": "exprHash", "value": "f755b3fc8'
    'eba"}], "description": "Bats", "logicalType": "string", "name": "batting_hand", "required": false}, '
    '{"customProperties": [{"property": "kind", "value": "field"}, {"property": "exprHash", "value": "1a3'
    'e9d68531f"}], "description": "Position", "logicalType": "string", "name": "position", "required": fa'
    'lse}, {"customProperties": [{"property": "kind", "value": "field"}, {"property": "exprHash", "value"'
    ': "a6a8b4c76709"}], "description": "Team name.", "logicalType": "string", "name": "team_name", "requ'
    'ired": false}, {"customProperties": [{"property": "kind", "value": "field"}, {"property": "exprHash"'
    ', "value": "58a3bb6e9d5f"}], "description": "League (North or South).", "logicalType": "string", "na'
    'me": "league", "required": false}, {"customProperties": [{"property": "kind", "value": "field"}, {"p'
    'roperty": "exprHash", "value": "7f824915493f"}], "description": "City", "logicalType": "string", "na'
    'me": "city", "required": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"p'
    'roperty": "exprHash", "value": "2c693824fe80"}], "description": "Games", "logicalType": "integer", "'
    'name": "games", "required": false}, {"customProperties": [{"property": "kind", "value": "measure"}, '
    '{"property": "exprHash", "value": "049e5740b70b"}], "description": "At Bats", "logicalType": "intege'
    'r", "name": "at_bats", "required": false}, {"customProperties": [{"property": "kind", "value": "meas'
    'ure"}, {"property": "exprHash", "value": "c37ec28232c8"}], "description": "Hits", "logicalType": "in'
    'teger", "name": "hits", "required": false}, {"customProperties": [{"property": "kind", "value": "mea'
    'sure"}, {"property": "exprHash", "value": "ff5094558c95"}], "description": "Doubles", "logicalType":'
    ' "integer", "name": "doubles", "required": false}, {"customProperties": [{"property": "kind", "value'
    '": "measure"}, {"property": "exprHash", "value": "f713219e2a51"}], "description": "Triples", "logica'
    'lType": "integer", "name": "triples", "required": false}, {"customProperties": [{"property": "kind",'
    ' "value": "measure"}, {"property": "exprHash", "value": "5c00cce45643"}], "description": "Home Runs"'
    ', "logicalType": "integer", "name": "home_runs", "required": false}, {"customProperties": [{"propert'
    'y": "kind", "value": "measure"}, {"property": "exprHash", "value": "7931f243ed37"}], "description": '
    '"RBI", "logicalType": "integer", "name": "rbi", "required": false}, {"customProperties": [{"property'
    '": "kind", "value": "measure"}, {"property": "exprHash", "value": "49499ba878d5"}], "description": "'
    'Walks", "logicalType": "integer", "name": "walks", "required": false}, {"customProperties": [{"prope'
    'rty": "kind", "value": "measure"}, {"property": "exprHash", "value": "b8058a5fd825"}], "description"'
    ': "Strikeouts", "logicalType": "integer", "name": "strikeouts", "required": false}, {"customProperti'
    'es": [{"property": "kind", "value": "measure"}, {"property": "exprHash", "value": "d86519884e33"}], '
    '"description": "Stolen Bases", "logicalType": "integer", "name": "stolen_bases", "required": false},'
    ' {"customProperties": [{"property": "kind", "value": "measure"}, {"property": "exprHash", "value": "'
    'd6c4f4e83168"}], "description": "Summed hits / summed at bats. Never average player averages.", "log'
    'icalType": "number", "name": "batting_avg", "required": false}, {"customProperties": [{"property": "'
    'kind", "value": "measure"}, {"property": "exprHash", "value": "bad3b89f4bc8"}], "description": "On-B'
    'ase Percentage (approx)", "logicalType": "number", "name": "on_base_pct_approx", "required": false},'
    ' {"customProperties": [{"property": "kind", "value": "measure"}, {"property": "exprHash", "value": "'
    'e895bedadcdb"}], "description": "Slugging Percentage", "logicalType": "number", "name": "slugging_pc'
    't", "required": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property":'
    ' "exprHash", "value": "d09e5a2b8d9e"}], "description": "OPS (approx)", "logicalType": "number", "nam'
    'e": "ops_approx", "required": false}, {"customProperties": [{"property": "kind", "value": "measure"}'
    ', {"property": "exprHash", "value": "aa8e24520508"}], "description": "Home Run Rate (per at bat)", "'
    'logicalType": "number", "name": "home_run_rate", "required": false}, {"customProperties": [{"propert'
    'y": "kind", "value": "measure"}, {"property": "exprHash", "value": "f9aadfad8d87"}], "description": '
    '"Distinct players. Non-additive.", "logicalType": "integer", "name": "batters", "required": false}],'
    ' "quality": [{"description": "Home runs by team in the metric view equal the sum from the gold fact.'
    '", "dimension": "accuracy", "engine": "fbl-semantic-checks", "implementation": {"compare_sql": "SELE'
    'CT t.team_name, SUM(f.hr) AS home_runs FROM gold.fbl_fact_batting f JOIN gold.fbl_dim_team t ON f.te'
    'am_id = t.team_id GROUP BY t.team_name", "dimensions": ["team_name"], "measures": ["home_runs"], "te'
    'st_type": "reconcile", "tolerance": 0}, "name": "SEM-FBLB-001", "severity": "error", "type": "custom'
    '"}, {"description": "Batting average by league equals summed hits over summed at bats from the gold '
    'fact (no average of averages).", "dimension": "accuracy", "engine": "fbl-semantic-checks", "implemen'
    'tation": {"compare_sql": "SELECT t.league, SUM(f.h) / NULLIF(SUM(f.ab), 0) AS batting_avg FROM gold.'
    'fbl_fact_batting f JOIN gold.fbl_dim_team t ON f.team_id = t.team_id GROUP BY t.league", "dimensions'
    '": ["league"], "measures": ["batting_avg"], "test_type": "reconcile", "tolerance": 1e-09}, "name": "'
    'SEM-FBLB-002", "severity": "error", "type": "custom"}, {"description": "Distinct batters per team eq'
    'uals the distinct players in the fact (distinct counts are not additive).", "dimension": "accuracy",'
    ' "engine": "fbl-semantic-checks", "implementation": {"compare_sql": "SELECT t.team_name, COUNT(DISTI'
    'NCT f.player_id) AS batters FROM gold.fbl_fact_batting f JOIN gold.fbl_dim_team t ON f.team_id = t.t'
    'eam_id GROUP BY t.team_name", "dimensions": ["team_name"], "measures": ["batters"], "test_type": "re'
    'concile", "tolerance": 0}, "name": "SEM-FBLB-003", "severity": "error", "type": "custom"}, {"descrip'
    'tion": "League-wide totals match the fact table.", "dimension": "accuracy", "engine": "fbl-semantic-'
    'checks", "implementation": {"compare_sql": "SELECT SUM(hr) AS home_runs, SUM(h) AS hits, SUM(ab) AS '
    'at_bats FROM gold.fbl_fact_batting", "dimensions": [], "measures": ["home_runs", "hits", "at_bats"],'
    ' "test_type": "reconcile", "tolerance": 0}, "name": "SEM-FBLB-004", "severity": "error", "type": "cu'
    'stom"}, {"description": "Rates stay between 0 and 1 for every team and season.", "dimension": "accur'
    'acy", "engine": "fbl-semantic-checks", "implementation": {"bounds": {"batting_avg": [0, 1], "home_ru'
    'n_rate": [0, 1], "on_base_pct_approx": [0, 1], "slugging_pct": [0, 4]}, "dimensions": ["team_name", '
    '"season"], "test_type": "bounds"}, "name": "SEM-FBLB-005", "severity": "error", "type": "custom"}]}]'
    ', "slaProperties": [{"element": "mv_fbl_batting", "property": "freshness", "unit": "h", "value": 24}'
    ', {"element": "mv_fbl_batting", "property": "frequency", "unit": "d", "value": 1}], "status": "activ'
    'e", "tags": ["fbl", "semantic"], "version": "1.0.0"}, "id": "fbl.semantic.mv_fbl_batting"}, {"checks'
    'um": "4cdae25404429478", "contract": {"apiVersion": "v3.1.0", "customProperties": [{"property": "lay'
    'er", "value": "semantic"}, {"property": "owner", "value": "Analytics Engineering"}, {"property": "st'
    'rictTypes", "value": false}], "dataProduct": "fbl", "description": {"limitations": "Roster only: one'
    ' row per player in the dimension. Contains no performance statistics. Personal data (birth date) is '
    'not exposed here.", "purpose": "Governed roster metrics for the Fictional Baseball League: headcount'
    ', average height and weight, switch hitters, by team, league, position and handedness.", "usage": "S'
    'ELECT dim, MEASURE(measure) FROM semantic.mv_fbl_roster GROUP BY dim."}, "domain": "baseball_operati'
    'ons", "id": "fbl.semantic.mv_fbl_roster", "kind": "DataContract", "name": "mv_fbl_roster (semantic)"'
    ', "schema": [{"customProperties": [{"property": "definition", "value": "sql/fbl/metric_views/mv_fbl_'
    'roster.sql"}, {"property": "dependsOn", "value": ["gold.fbl_dim_player", "gold.fbl_dim_team"]}, {"pr'
    'operty": "consumers", "value": ["bi_dashboards"]}], "description": "Governed roster metrics for the '
    'Fictional Baseball League: headcount, average height and weight, switch hitters, by team, league, po'
    'sition and handedness.", "logicalType": "object", "name": "mv_fbl_roster", "physicalName": "semantic'
    '.mv_fbl_roster", "physicalType": "view", "properties": [{"customProperties": [{"property": "kind", "'
    'value": "field"}, {"property": "exprHash", "value": "a6a8b4c76709"}], "description": "Team name.", "'
    'logicalType": "string", "name": "team_name", "required": false}, {"customProperties": [{"property": '
    '"kind", "value": "field"}, {"property": "exprHash", "value": "58a3bb6e9d5f"}], "description": "Leagu'
    'e (North or South).", "logicalType": "string", "name": "league", "required": false}, {"customPropert'
    'ies": [{"property": "kind", "value": "field"}, {"property": "exprHash", "value": "ff6877ec3c1f"}], "'
    'description": "Position", "logicalType": "string", "name": "position", "required": false}, {"customP'
    'roperties": [{"property": "kind", "value": "field"}, {"property": "exprHash", "value": "133a7855692e'
    '"}], "description": "Bats", "logicalType": "string", "name": "batting_hand", "required": false}, {"c'
    'ustomProperties": [{"property": "kind", "value": "field"}, {"property": "exprHash", "value": "c6e578'
    '13abc5"}], "description": "Throws", "logicalType": "string", "name": "throwing_hand", "required": fa'
    'lse}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property": "exprHash", "valu'
    'e": "5c14c3cbd2d3"}], "description": "Number of players. Count of rows in the player dimension.", "l'
    'ogicalType": "integer", "name": "players", "required": false}, {"customProperties": [{"property": "k'
    'ind", "value": "measure"}, {"property": "exprHash", "value": "bb63b5fb2561"}], "description": "Avera'
    'ge Height (in)", "logicalType": "number", "name": "avg_height_in", "required": false}, {"customPrope'
    'rties": [{"property": "kind", "value": "measure"}, {"property": "exprHash", "value": "7235c822efd3"}'
    '], "description": "Average Weight (lb)", "logicalType": "number", "name": "avg_weight_lb", "required'
    '": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property": "exprHash", '
    '"value": "465e950936e5"}], "description": "Switch Hitters", "logicalType": "integer", "name": "switc'
    'h_hitters", "required": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"pr'
    'operty": "exprHash", "value": "260af457735d"}], "description": "Distinct Teams", "logicalType": "int'
    'eger", "name": "teams", "required": false}], "quality": [{"description": "Players per team equals th'
    'e player dimension counts.", "dimension": "accuracy", "engine": "fbl-semantic-checks", "implementati'
    'on": {"compare_sql": "SELECT t.team_name, COUNT(*) AS players FROM gold.fbl_dim_player p JOIN gold.f'
    'bl_dim_team t ON p.team_id = t.team_id GROUP BY t.team_name", "dimensions": ["team_name"], "measures'
    '": ["players"], "test_type": "reconcile", "tolerance": 0}, "name": "SEM-FBLR-001", "severity": "erro'
    'r", "type": "custom"}, {"description": "Average height is plausible for every team and position.", "'
    'dimension": "accuracy", "engine": "fbl-semantic-checks", "implementation": {"bounds": {"avg_height_i'
    'n": [60, 84]}, "dimensions": ["team_name", "position"], "test_type": "bounds"}, "name": "SEM-FBLR-00'
    '2", "severity": "error", "type": "custom"}, {"description": "Average weight is plausible. WARNING le'
    'vel: silver only warns about odd weights (PLR-008), so this test surfaces any that reached gold.", "'
    'dimension": "accuracy", "engine": "fbl-semantic-checks", "implementation": {"bounds": {"avg_weight_l'
    'b": [120, 350]}, "dimensions": ["team_name", "position"], "test_type": "bounds"}, "name": "SEM-FBLR-'
    '004", "severity": "warning", "type": "custom"}, {"description": "Total players equals the player dim'
    'ension row count.", "dimension": "accuracy", "engine": "fbl-semantic-checks", "implementation": {"co'
    'mpare_sql": "SELECT COUNT(*) AS players FROM gold.fbl_dim_player", "dimensions": [], "measures": ["p'
    'layers"], "test_type": "reconcile", "tolerance": 0}, "name": "SEM-FBLR-003", "severity": "error", "t'
    'ype": "custom"}]}], "slaProperties": [{"element": "mv_fbl_roster", "property": "freshness", "unit": '
    '"h", "value": 24}, {"element": "mv_fbl_roster", "property": "frequency", "unit": "d", "value": 1}], '
    '"status": "active", "tags": ["fbl", "semantic"], "version": "1.0.0"}, "id": "fbl.semantic.mv_fbl_ros'
    'ter"}, {"checksum": "a00fcf13e915498b", "contract": {"apiVersion": "v3.1.0", "customProperties": [{"'
    'property": "layer", "value": "semantic"}, {"property": "owner", "value": "Analytics Engineering"}, {'
    '"property": "strictTypes", "value": false}], "dataProduct": "lahman", "description": {"limitations":'
    ' "NULL means not recorded (e.g. no sacrifice flies before 1954), not zero. Distinct counts are not a'
    'dditive.", "purpose": "Governed batting metrics for MLB 1871-2020 (Lahman). Ratios are computed from'
    ' summed components.", "usage": "SELECT dim, MEASURE(measure) FROM semantic.mv_batting GROUP BY dim."'
    '}, "domain": "baseball_operations", "id": "lahman.semantic.mv_batting", "kind": "DataContract", "nam'
    'e": "mv_batting (semantic)", "schema": [{"customProperties": [{"property": "definition", "value": "s'
    'ql/metric_views/10_mv_batting.sql"}, {"property": "dependsOn", "value": ["gold.fact_batting_season",'
    ' "gold.dim_player", "gold.dim_team_season", "gold.dim_season"]}, {"property": "consumers", "value": '
    '["bi_dashboards", "genie_space_mlb"]}], "description": "Governed batting metrics for MLB 1871-2020 ('
    'Lahman). Ratios are computed from summed components.", "logicalType": "object", "name": "mv_batting"'
    ', "physicalName": "semantic.mv_batting", "physicalType": "view", "properties": [{"customProperties":'
    ' [{"property": "kind", "value": "field"}, {"property": "exprHash", "value": "18c81f0acf57"}], "descr'
    'iption": "Season", "logicalType": "integer", "name": "season", "required": false}, {"customPropertie'
    's": [{"property": "kind", "value": "field"}, {"property": "exprHash", "value": "4854e3a4b48a"}], "de'
    'scription": "Decade", "logicalType": "integer", "name": "decade", "required": false}, {"customProper'
    'ties": [{"property": "kind", "value": "field"}, {"property": "exprHash", "value": "87a969fc3eef"}], '
    '"description": "Baseball Era", "logicalType": "string", "name": "baseball_era", "required": false}, '
    '{"customProperties": [{"property": "kind", "value": "field"}, {"property": "exprHash", "value": "d28'
    'd7eea452f"}], "description": "Player", "logicalType": "string", "name": "player_name", "required": f'
    'alse}, {"customProperties": [{"property": "kind", "value": "field"}, {"property": "exprHash", "value'
    '": "42a681b4211a"}], "description": "Bats", "logicalType": "string", "name": "batting_hand", "requir'
    'ed": false}, {"customProperties": [{"property": "kind", "value": "field"}, {"property": "exprHash", '
    '"value": "9b4ceee9c61a"}], "description": "Birth Country", "logicalType": "string", "name": "birth_c'
    'ountry", "required": false}, {"customProperties": [{"property": "kind", "value": "field"}, {"propert'
    'y": "exprHash", "value": "a6a8b4c76709"}], "description": "Team", "logicalType": "string", "name": "'
    'team_name", "required": false}, {"customProperties": [{"property": "kind", "value": "field"}, {"prop'
    'erty": "exprHash", "value": "a0408473bec7"}], "description": "Franchise", "logicalType": "string", "'
    'name": "franchise", "required": false}, {"customProperties": [{"property": "kind", "value": "field"}'
    ', {"property": "exprHash", "value": "5a8a79f3c146"}], "description": "League", "logicalType": "strin'
    'g", "name": "league", "required": false}, {"customProperties": [{"property": "kind", "value": "field'
    '"}, {"property": "exprHash", "value": "9d1afb56297f"}], "description": "Division", "logicalType": "s'
    'tring", "name": "division", "required": false}, {"customProperties": [{"property": "kind", "value": '
    '"measure"}, {"property": "exprHash", "value": "2c693824fe80"}], "description": "Games", "logicalType'
    '": "integer", "name": "games", "required": false}, {"customProperties": [{"property": "kind", "value'
    '": "measure"}, {"property": "exprHash", "value": "049e5740b70b"}], "description": "At Bats", "logica'
    'lType": "integer", "name": "at_bats", "required": false}, {"customProperties": [{"property": "kind",'
    ' "value": "measure"}, {"property": "exprHash", "value": "254ccffd6043"}], "description": "Plate Appe'
    'arances", "logicalType": "integer", "name": "plate_appearances", "required": false}, {"customPropert'
    'ies": [{"property": "kind", "value": "measure"}, {"property": "exprHash", "value": "b88fa486fc3e"}],'
    ' "description": "Runs", "logicalType": "integer", "name": "runs", "required": false}, {"customProper'
    'ties": [{"property": "kind", "value": "measure"}, {"property": "exprHash", "value": "c37ec28232c8"}]'
    ', "description": "Hits", "logicalType": "integer", "name": "hits", "required": false}, {"customPrope'
    'rties": [{"property": "kind", "value": "measure"}, {"property": "exprHash", "value": "5c00cce45643"}'
    '], "description": "Home Runs", "logicalType": "integer", "name": "home_runs", "required": false}, {"'
    'customProperties": [{"property": "kind", "value": "measure"}, {"property": "exprHash", "value": "793'
    '1f243ed37"}], "description": "RBI", "logicalType": "integer", "name": "rbi", "required": false}, {"c'
    'ustomProperties": [{"property": "kind", "value": "measure"}, {"property": "exprHash", "value": "d865'
    '19884e33"}], "description": "Stolen Bases", "logicalType": "integer", "name": "stolen_bases", "requi'
    'red": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property": "exprHash'
    '", "value": "49499ba878d5"}], "description": "Walks", "logicalType": "integer", "name": "walks", "re'
    'quired": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property": "exprH'
    'ash", "value": "b8058a5fd825"}], "description": "Strikeouts", "logicalType": "integer", "name": "str'
    'ikeouts", "required": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"prop'
    'erty": "exprHash", "value": "d6c4f4e83168"}], "description": "Hits divided by at bats. Never average'
    ' player averages: always divide summed hits by summed at bats.", "logicalType": "number", "name": "b'
    'atting_avg", "required": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"p'
    'roperty": "exprHash", "value": "04ebab5c3420"}], "description": "On-Base Percentage", "logicalType":'
    ' "number", "name": "on_base_pct", "required": false}, {"customProperties": [{"property": "kind", "va'
    'lue": "measure"}, {"property": "exprHash", "value": "e895bedadcdb"}], "description": "Slugging Perce'
    'ntage", "logicalType": "number", "name": "slugging_pct", "required": false}, {"customProperties": [{'
    '"property": "kind", "value": "measure"}, {"property": "exprHash", "value": "05d5ae136aba"}], "descri'
    'ption": "Composed measure: built from two earlier measures with MEASURE(). Change OBP once, OPS foll'
    'ows.", "logicalType": "number", "name": "ops", "required": false}, {"customProperties": [{"property"'
    ': "kind", "value": "measure"}, {"property": "exprHash", "value": "29c731829bbf"}], "description": "I'
    'solated Power", "logicalType": "number", "name": "isolated_power", "required": false}, {"customPrope'
    'rties": [{"property": "kind", "value": "measure"}, {"property": "exprHash", "value": "578ea68db733"}'
    '], "description": "Home Run Rate", "logicalType": "number", "name": "home_run_rate", "required": fal'
    'se}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property": "exprHash", "value'
    '": "c829739abd75"}], "description": "Strikeout Rate", "logicalType": "number", "name": "strikeout_ra'
    'te", "required": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property"'
    ': "exprHash", "value": "8706e47a2197"}], "description": "Walk Rate", "logicalType": "number", "name"'
    ': "walk_rate", "required": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {'
    '"property": "exprHash", "value": "4244c4e7ed04"}], "description": "Stolen Base Success Rate", "logic'
    'alType": "number", "name": "stolen_base_success_rate", "required": false}, {"customProperties": [{"p'
    'roperty": "kind", "value": "measure"}, {"property": "exprHash", "value": "7a9129d1f32e"}], "descript'
    'ion": "Non-additive: distinct counts cannot be summed across groups. The metric view handles that fo'
    'r you.", "logicalType": "integer", "name": "batters", "required": false}, {"customProperties": [{"pr'
    'operty": "kind", "value": "measure"}, {"property": "exprHash", "value": "bcaf74032868"}], "descripti'
    'on": "Filtered measure example: only rows with at least 400 at bats.", "logicalType": "integer", "na'
    'me": "home_runs_regular_stints", "required": false}, {"customProperties": [{"property": "kind", "val'
    'ue": "measure"}, {"property": "exprHash", "value": "5c00cce45643"}], "description": "Window measure:'
    ' running total ordered by season. Group by season to see the curve.", "logicalType": "integer", "nam'
    'e": "cumulative_home_runs", "required": false}], "quality": [{"description": "Bonds 2001 returns the'
    ' record-book home runs and batting average.", "dimension": "accuracy", "engine": "fbl-semantic-check'
    's", "implementation": {"dimensions": ["player_name", "season"], "expect": [{"batting_avg": 0.3277, "'
    'home_runs": 73, "player_name": "Barry Bonds", "season": 2001}], "measures": ["home_runs", "batting_a'
    'vg"], "test_type": "expect", "tolerance": 0.0005, "where": "player_name = \'Barry Bonds\' AND season ='
    ' 2001"}, "name": "SEM-LB-001", "severity": "error", "type": "custom"}, {"description": "Home runs by'
    ' season equal the sum from the gold fact.", "dimension": "accuracy", "engine": "fbl-semantic-checks"'
    ', "implementation": {"compare_sql": "SELECT season, SUM(hr) AS home_runs FROM gold.fact_batting_seas'
    'on GROUP BY season", "dimensions": ["season"], "measures": ["home_runs"], "test_type": "reconcile", '
    '"tolerance": 0}, "name": "SEM-LB-002", "severity": "error", "type": "custom"}, {"description": "2001'
    ' league-wide home runs match the raw Lahman file.", "dimension": "accuracy", "engine": "fbl-semantic'
    '-checks", "implementation": {"dimensions": ["season"], "expect": [{"home_runs": 5458, "season": 2001'
    '}], "measures": ["home_runs"], "test_type": "expect", "tolerance": 0, "where": "season = 2001"}, "na'
    'me": "SEM-LB-003", "severity": "error", "type": "custom"}, {"description": "Batting average is betwe'
    'en 0 and 1 for every season and league.", "dimension": "accuracy", "engine": "fbl-semantic-checks", '
    '"implementation": {"bounds": {"batting_avg": [0, 1]}, "dimensions": ["season", "league"], "test_type'
    '": "bounds"}, "name": "SEM-LB-004", "severity": "error", "type": "custom"}]}], "slaProperties": [{"e'
    'lement": "mv_batting", "property": "frequency", "unit": "d", "value": 1}], "status": "active", "tags'
    '": ["lahman", "semantic"], "version": "1.0.0"}, "id": "lahman.semantic.mv_batting"}, {"checksum": "b'
    'c735ce2b8a20ec7", "contract": {"apiVersion": "v3.1.0", "customProperties": [{"property": "layer", "v'
    'alue": "semantic"}, {"property": "owner", "value": "Analytics Engineering"}, {"property": "strictTyp'
    'es", "value": false}], "dataProduct": "lahman", "description": {"limitations": "Older seasons have m'
    'issing strikeout and walk detail. Distinct counts are not additive.", "purpose": "Governed pitching '
    'metrics for MLB 1871-2020 (Lahman). Rates use outs (ipouts / 3), not decimal innings.", "usage": "SE'
    'LECT dim, MEASURE(measure) FROM semantic.mv_pitching GROUP BY dim."}, "domain": "baseball_operations'
    '", "id": "lahman.semantic.mv_pitching", "kind": "DataContract", "name": "mv_pitching (semantic)", "s'
    'chema": [{"customProperties": [{"property": "definition", "value": "sql/metric_views/20_mv_pitching.'
    'sql"}, {"property": "dependsOn", "value": ["gold.fact_pitching_season", "gold.dim_player", "gold.dim'
    '_team_season", "gold.dim_season"]}, {"property": "consumers", "value": ["bi_dashboards", "genie_spac'
    'e_mlb"]}], "description": "Governed pitching metrics for MLB 1871-2020 (Lahman). Rates use outs (ipo'
    'uts / 3), not decimal innings.", "logicalType": "object", "name": "mv_pitching", "physicalName": "se'
    'mantic.mv_pitching", "physicalType": "view", "properties": [{"customProperties": [{"property": "kind'
    '", "value": "field"}, {"property": "exprHash", "value": "18c81f0acf57"}], "description": "Season", "'
    'logicalType": "integer", "name": "season", "required": false}, {"customProperties": [{"property": "k'
    'ind", "value": "field"}, {"property": "exprHash", "value": "4854e3a4b48a"}], "description": "Decade"'
    ', "logicalType": "integer", "name": "decade", "required": false}, {"customProperties": [{"property":'
    ' "kind", "value": "field"}, {"property": "exprHash", "value": "87a969fc3eef"}], "description": "Base'
    'ball Era", "logicalType": "string", "name": "baseball_era", "required": false}, {"customProperties":'
    ' [{"property": "kind", "value": "field"}, {"property": "exprHash", "value": "d28d7eea452f"}], "descr'
    'iption": "Pitcher", "logicalType": "string", "name": "player_name", "required": false}, {"customProp'
    'erties": [{"property": "kind", "value": "field"}, {"property": "exprHash", "value": "b8d3e296ee47"}]'
    ', "description": "Throws", "logicalType": "string", "name": "throws", "required": false}, {"customPr'
    'operties": [{"property": "kind", "value": "field"}, {"property": "exprHash", "value": "a6a8b4c76709"'
    '}], "description": "Team", "logicalType": "string", "name": "team_name", "required": false}, {"custo'
    'mProperties": [{"property": "kind", "value": "field"}, {"property": "exprHash", "value": "a0408473be'
    'c7"}], "description": "Franchise", "logicalType": "string", "name": "franchise", "required": false},'
    ' {"customProperties": [{"property": "kind", "value": "field"}, {"property": "exprHash", "value": "5a'
    '8a79f3c146"}], "description": "League", "logicalType": "string", "name": "league", "required": false'
    '}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property": "exprHash", "value":'
    ' "2c693824fe80"}], "description": "Games", "logicalType": "integer", "name": "games", "required": fa'
    'lse}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property": "exprHash", "valu'
    'e": "4bdc328b400f"}], "description": "Games Started", "logicalType": "integer", "name": "games_start'
    'ed", "required": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property"'
    ': "exprHash", "value": "30339175bb51"}], "description": "Wins", "logicalType": "integer", "name": "w'
    'ins", "required": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property'
    '": "exprHash", "value": "f6c15137548e"}], "description": "Losses", "logicalType": "integer", "name":'
    ' "losses", "required": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"pro'
    'perty": "exprHash", "value": "2635cd36f639"}], "description": "Saves", "logicalType": "integer", "na'
    'me": "saves", "required": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"'
    'property": "exprHash", "value": "ddae53bfcba2"}], "description": "Complete Games", "logicalType": "i'
    'nteger", "name": "complete_games", "required": false}, {"customProperties": [{"property": "kind", "v'
    'alue": "measure"}, {"property": "exprHash", "value": "a9d84c979c60"}], "description": "Shutouts", "l'
    'ogicalType": "integer", "name": "shutouts", "required": false}, {"customProperties": [{"property": "'
    'kind", "value": "measure"}, {"property": "exprHash", "value": "b8058a5fd825"}], "description": "Stri'
    'keouts", "logicalType": "integer", "name": "strikeouts", "required": false}, {"customProperties": [{'
    '"property": "kind", "value": "measure"}, {"property": "exprHash", "value": "49499ba878d5"}], "descri'
    'ption": "Walks", "logicalType": "integer", "name": "walks", "required": false}, {"customProperties":'
    ' [{"property": "kind", "value": "measure"}, {"property": "exprHash", "value": "c37ec28232c8"}], "des'
    'cription": "Hits Allowed", "logicalType": "integer", "name": "hits_allowed", "required": false}, {"c'
    'ustomProperties": [{"property": "kind", "value": "measure"}, {"property": "exprHash", "value": "5c00'
    'cce45643"}], "description": "Home Runs Allowed", "logicalType": "integer", "name": "home_runs_allowe'
    'd", "required": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property":'
    ' "exprHash", "value": "891824f189c3"}], "description": "Earned Runs", "logicalType": "integer", "nam'
    'e": "earned_runs", "required": false}, {"customProperties": [{"property": "kind", "value": "measure"'
    '}, {"property": "exprHash", "value": "c18373fbce26"}], "description": "Innings Pitched", "logicalTyp'
    'e": "number", "name": "innings_pitched", "required": false}, {"customProperties": [{"property": "kin'
    'd", "value": "measure"}, {"property": "exprHash", "value": "c624911c4ccc"}], "description": "Earned '
    'runs per nine innings = 27 x ER / outs recorded.", "logicalType": "number", "name": "era", "required'
    '": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property": "exprHash", '
    '"value": "9dd00f25c5ad"}], "description": "WHIP", "logicalType": "number", "name": "whip", "required'
    '": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property": "exprHash", '
    '"value": "0f579d443eeb"}], "description": "Strikeouts per 9", "logicalType": "number", "name": "stri'
    'keouts_per_9", "required": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {'
    '"property": "exprHash", "value": "24d7ddbabf3d"}], "description": "Walks per 9", "logicalType": "num'
    'ber", "name": "walks_per_9", "required": false}, {"customProperties": [{"property": "kind", "value":'
    ' "measure"}, {"property": "exprHash", "value": "d4219e3b6d12"}], "description": "Home Runs per 9", "'
    'logicalType": "number", "name": "home_runs_per_9", "required": false}, {"customProperties": [{"prope'
    'rty": "kind", "value": "measure"}, {"property": "exprHash", "value": "263d76edffff"}], "description"'
    ': "Strikeout-to-Walk Ratio", "logicalType": "number", "name": "strikeout_to_walk_ratio", "required":'
    ' false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property": "exprHash", "v'
    'alue": "5178c874383e"}], "description": "Win Percentage", "logicalType": "number", "name": "win_pct"'
    ', "required": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property": "'
    'exprHash", "value": "7a9129d1f32e"}], "description": "Distinct Pitchers", "logicalType": "integer", '
    '"name": "pitchers", "required": false}], "quality": [{"description": "2001 league-wide strikeouts ma'
    'tch the raw Lahman file.", "dimension": "accuracy", "engine": "fbl-semantic-checks", "implementation'
    '": {"dimensions": ["season"], "expect": [{"season": 2001, "strikeouts": 32404}], "measures": ["strik'
    'eouts"], "test_type": "expect", "tolerance": 0, "where": "season = 2001"}, "name": "SEM-LP-001", "se'
    'verity": "error", "type": "custom"}, {"description": "Strikeouts by season equal the sum from the go'
    'ld fact.", "dimension": "accuracy", "engine": "fbl-semantic-checks", "implementation": {"compare_sql'
    '": "SELECT season, SUM(so) AS strikeouts FROM gold.fact_pitching_season GROUP BY season", "dimension'
    's": ["season"], "measures": ["strikeouts"], "test_type": "reconcile", "tolerance": 0}, "name": "SEM-'
    'LP-002", "severity": "error", "type": "custom"}, {"description": "Win percentage is between 0 and 1 '
    'for every season.", "dimension": "accuracy", "engine": "fbl-semantic-checks", "implementation": {"bo'
    'unds": {"win_pct": [0, 1]}, "dimensions": ["season"], "test_type": "bounds"}, "name": "SEM-LP-003", '
    '"severity": "error", "type": "custom"}]}], "slaProperties": [{"element": "mv_pitching", "property": '
    '"frequency", "unit": "d", "value": 1}], "status": "active", "tags": ["lahman", "semantic"], "version'
    '": "1.0.0"}, "id": "lahman.semantic.mv_pitching"}, {"checksum": "6594067b5ed280d8", "contract": {"ap'
    'iVersion": "v3.1.0", "customProperties": [{"property": "layer", "value": "semantic"}, {"property": "'
    'owner", "value": "Analytics Engineering"}, {"property": "strictTypes", "value": false}], "dataProduc'
    't": "lahman", "description": {"limitations": "Payroll exists from 1985 only and is NULL before. Cost'
    ' per win only counts seasons with payroll.", "purpose": "Governed team-season metrics (results, run '
    'environment, attendance, payroll) for MLB 1871-2020 (Lahman).", "usage": "SELECT dim, MEASURE(measur'
    'e) FROM semantic.mv_team_season GROUP BY dim."}, "domain": "baseball_operations", "id": "lahman.sema'
    'ntic.mv_team_season", "kind": "DataContract", "name": "mv_team_season (semantic)", "schema": [{"cust'
    'omProperties": [{"property": "definition", "value": "sql/metric_views/30_mv_team_season.sql"}, {"pro'
    'perty": "dependsOn", "value": ["gold.fact_team_season", "gold.dim_team_season", "gold.dim_season"]},'
    ' {"property": "consumers", "value": ["bi_dashboards", "genie_space_mlb"]}], "description": "Governed'
    ' team-season metrics (results, run environment, attendance, payroll) for MLB 1871-2020 (Lahman).", "'
    'logicalType": "object", "name": "mv_team_season", "physicalName": "semantic.mv_team_season", "physic'
    'alType": "view", "properties": [{"customProperties": [{"property": "kind", "value": "field"}, {"prop'
    'erty": "exprHash", "value": "18c81f0acf57"}], "description": "Season", "logicalType": "integer", "na'
    'me": "season", "required": false}, {"customProperties": [{"property": "kind", "value": "field"}, {"p'
    'roperty": "exprHash", "value": "4854e3a4b48a"}], "description": "Decade", "logicalType": "integer", '
    '"name": "decade", "required": false}, {"customProperties": [{"property": "kind", "value": "field"}, '
    '{"property": "exprHash", "value": "87a969fc3eef"}], "description": "Baseball Era", "logicalType": "s'
    'tring", "name": "baseball_era", "required": false}, {"customProperties": [{"property": "kind", "valu'
    'e": "field"}, {"property": "exprHash", "value": "a6a8b4c76709"}], "description": "Team", "logicalTyp'
    'e": "string", "name": "team_name", "required": false}, {"customProperties": [{"property": "kind", "v'
    'alue": "field"}, {"property": "exprHash", "value": "a0408473bec7"}], "description": "Franchise", "lo'
    'gicalType": "string", "name": "franchise", "required": false}, {"customProperties": [{"property": "k'
    'ind", "value": "field"}, {"property": "exprHash", "value": "5a8a79f3c146"}], "description": "League"'
    ', "logicalType": "string", "name": "league", "required": false}, {"customProperties": [{"property": '
    '"kind", "value": "field"}, {"property": "exprHash", "value": "9d1afb56297f"}], "description": "Divis'
    'ion", "logicalType": "string", "name": "division", "required": false}, {"customProperties": [{"prope'
    'rty": "kind", "value": "field"}, {"property": "exprHash", "value": "99484b3203d3"}], "description": '
    '"Ballpark", "logicalType": "string", "name": "ballpark", "required": false}, {"customProperties": [{'
    '"property": "kind", "value": "field"}, {"property": "exprHash", "value": "05c67ee92ab2"}], "descript'
    'ion": "Won World Series", "logicalType": "boolean", "name": "won_world_series", "required": false}, '
    '{"customProperties": [{"property": "kind", "value": "measure"}, {"property": "exprHash", "value": "5'
    'c14c3cbd2d3"}], "description": "Team Seasons", "logicalType": "integer", "name": "team_seasons", "re'
    'quired": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property": "exprH'
    'ash", "value": "2c693824fe80"}], "description": "Games", "logicalType": "integer", "name": "games", '
    '"required": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property": "ex'
    'prHash", "value": "30339175bb51"}], "description": "Wins", "logicalType": "integer", "name": "wins",'
    ' "required": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property": "e'
    'xprHash", "value": "f6c15137548e"}], "description": "Losses", "logicalType": "integer", "name": "los'
    'ses", "required": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property'
    '": "exprHash", "value": "5178c874383e"}], "description": "Win Percentage", "logicalType": "number", '
    '"name": "win_pct", "required": false}, {"customProperties": [{"property": "kind", "value": "measure"'
    '}, {"property": "exprHash", "value": "b88fa486fc3e"}], "description": "Runs Scored", "logicalType": '
    '"integer", "name": "runs_scored", "required": false}, {"customProperties": [{"property": "kind", "va'
    'lue": "measure"}, {"property": "exprHash", "value": "f9a948a63f02"}], "description": "Runs Allowed",'
    ' "logicalType": "integer", "name": "runs_allowed", "required": false}, {"customProperties": [{"prope'
    'rty": "kind", "value": "measure"}, {"property": "exprHash", "value": "7266756bf1a3"}], "description"'
    ': "Run Differential", "logicalType": "integer", "name": "run_differential", "required": false}, {"cu'
    'stomProperties": [{"property": "kind", "value": "measure"}, {"property": "exprHash", "value": "6b36f'
    'a6c9491"}], "description": "Expected win percentage from runs scored and allowed (exponent 1.83).", '
    '"logicalType": "number", "name": "pythagorean_win_pct", "required": false}, {"customProperties": [{"'
    'property": "kind", "value": "measure"}, {"property": "exprHash", "value": "21d7e67f740d"}], "descrip'
    'tion": "Win Pct Over Expected", "logicalType": "number", "name": "wins_over_expected", "required": f'
    'alse}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property": "exprHash", "val'
    'ue": "68ed0260374b"}], "description": "Attendance", "logicalType": "integer", "name": "attendance", '
    '"required": false}, {"customProperties": [{"property": "kind", "value": "measure"}, {"property": "ex'
    'prHash", "value": "4e13dece1f0b"}], "description": "Only counts home games where attendance was reco'
    'rded, so missing data does not drag the average down.", "logicalType": "number", "name": "avg_attend'
    'ance_per_home_game", "required": false}, {"customProperties": [{"property": "kind", "value": "measur'
    'e"}, {"property": "exprHash", "value": "e14b9921dc17"}], "description": "Sum of player salaries. NUL'
    'L before 1985.", "logicalType": "integer", "name": "payroll", "required": false}, {"customProperties'
    '": [{"property": "kind", "value": "measure"}, {"property": "exprHash", "value": "9874951be163"}], "d'
    'escription": "Wins in the denominator are restricted to seasons that have payroll, otherwise pre-198'
    '5 wins would flatter the ratio.", "logicalType": "number", "name": "cost_per_win", "required": false'
    '}], "quality": [{"description": "The 2001 Seattle Mariners show the record 116 wins and a +300 run d'
    'ifferential.", "dimension": "accuracy", "engine": "fbl-semantic-checks", "implementation": {"dimensi'
    'ons": ["team_name", "season"], "expect": [{"run_differential": 300, "season": 2001, "team_name": "Se'
    'attle Mariners", "wins": 116}], "measures": ["wins", "run_differential"], "test_type": "expect", "to'
    'lerance": 0, "where": "team_name = \'Seattle Mariners\' AND season = 2001"}, "name": "SEM-LT-001", "se'
    'verity": "error", "type": "custom"}, {"description": "Wins by season equal the sum from the gold fac'
    't.", "dimension": "accuracy", "engine": "fbl-semantic-checks", "implementation": {"compare_sql": "SE'
    'LECT season, SUM(w) AS wins FROM gold.fact_team_season GROUP BY season", "dimensions": ["season"], "'
    'measures": ["wins"], "test_type": "reconcile", "tolerance": 0}, "name": "SEM-LT-002", "severity": "e'
    'rror", "type": "custom"}, {"description": "Win percentage is between 0 and 1.", "dimension": "accura'
    'cy", "engine": "fbl-semantic-checks", "implementation": {"bounds": {"win_pct": [0, 1]}, "dimensions"'
    ': ["season"], "test_type": "bounds"}, "name": "SEM-LT-003", "severity": "error", "type": "custom"}]}'
    '], "slaProperties": [{"element": "mv_team_season", "property": "frequency", "unit": "d", "value": 1}'
    '], "status": "active", "tags": ["lahman", "semantic"], "version": "1.0.0"}, "id": "lahman.semantic.m'
    'v_team_season"}, {"checksum": "54095e6f00c88ee2", "contract": {"apiVersion": "v3.1.0", "customProper'
    'ties": [{"property": "layer", "value": "silver"}, {"property": "owner", "value": "Data Engineering"}'
    ', {"property": "dependsOn", "value": "fbl.source.batting"}], "dataProduct": "fbl", "description": {"'
    'limitations": "Rejected rows are NOT in this table; they are in governance.dq_quarantine with the re'
    'ason. Optional columns may still be NULL.", "purpose": "Typed, cleaned, deduplicated batting rows. E'
    'very row here has passed every quarantine rule below.", "usage": "Internal layer for data engineerin'
    'g. Consumers read gold or the semantic layer."}, "domain": "baseball_operations", "id": "fbl.silver.'
    'batting", "kind": "DataContract", "name": "FBL batting (silver)", "schema": [{"customProperties": [{'
    '"property": "entity", "value": "batting"}], "description": "Clean batting records.", "logicalType": '
    '"object", "name": "fbl_batting", "physicalName": "silver.fbl_batting", "physicalType": "table", "pro'
    'perties": [{"customProperties": [{"property": "standardize", "value": "upper"}], "description": "Pla'
    'yer. Part of the key.", "logicalType": "string", "name": "player_id", "physicalType": "STRING", "pri'
    'maryKey": true, "primaryKeyPosition": 1, "relationships": [{"to": "fbl_player.player_id", "type": "f'
    'oreignKey"}], "required": true}, {"description": "Season year. Part of the key.", "logicalType": "in'
    'teger", "name": "season", "physicalType": "INT", "primaryKey": true, "primaryKeyPosition": 2, "requi'
    'red": true}, {"customProperties": [{"property": "standardize", "value": "upper"}], "description": "T'
    'eam played for. Part of the key.", "logicalType": "string", "name": "team_id", "physicalType": "STRI'
    'NG", "primaryKey": true, "primaryKeyPosition": 3, "relationships": [{"to": "fbl_team.team_id", "type'
    '": "foreignKey"}], "required": true}, {"description": "Games played (0 to 162).", "logicalType": "in'
    'teger", "name": "g", "physicalType": "INT", "required": true}, {"description": "At bats.", "logicalT'
    'ype": "integer", "name": "ab", "physicalType": "INT", "required": true}, {"description": "Hits. Cann'
    'ot exceed at bats.", "logicalType": "integer", "name": "h", "physicalType": "INT", "required": true}'
    ', {"description": "Doubles.", "logicalType": "integer", "name": "doubles", "physicalType": "INT", "r'
    'equired": true}, {"description": "Triples.", "logicalType": "integer", "name": "triples", "physicalT'
    'ype": "INT", "required": true}, {"description": "Home runs. Cannot exceed hits.", "logicalType": "in'
    'teger", "name": "hr", "physicalType": "INT", "required": true}, {"description": "Runs batted in.", "'
    'logicalType": "integer", "name": "rbi", "physicalType": "INT", "required": true}, {"description": "W'
    'alks.", "logicalType": "integer", "name": "bb", "physicalType": "INT", "required": true}, {"descript'
    'ion": "Strikeouts.", "logicalType": "integer", "name": "so", "physicalType": "INT", "required": true'
    '}, {"description": "Stolen bases. Cannot be negative.", "logicalType": "integer", "name": "sb", "phy'
    'sicalType": "INT", "required": true}], "quality": [{"description": "Identical duplicate batting rows'
    ' are removed automatically (first copy kept)", "dimension": "uniqueness", "engine": "fbl-dq-engine",'
    ' "implementation": {"action": "dedupe", "columns": "*", "expression": "", "owner": "Data Engineering'
    '", "rule_type": "exact_duplicate"}, "name": "BAT-001", "severity": "warning", "type": "custom"}, {"d'
    'escription": "One batting line per player, season and team. Conflicting duplicates are rejected for '
    'human review", "dimension": "uniqueness", "engine": "fbl-dq-engine", "implementation": {"action": "q'
    'uarantine", "columns": ["player_id", "season", "team_id"], "expression": "", "owner": "Data Engineer'
    'ing", "rule_type": "unique_strict"}, "name": "BAT-002", "severity": "error", "type": "custom"}, {"de'
    'scription": "Required batting fields must be present (non-numeric text becomes NULL and is caught he'
    're)", "dimension": "completeness", "engine": "fbl-dq-engine", "implementation": {"action": "quaranti'
    'ne", "columns": ["player_id", "season", "team_id", "g", "ab", "h", "doubles", "triples", "hr", "rbi"'
    ', "bb", "so", "sb"], "expression": "", "owner": "Data Engineering", "rule_type": "not_null"}, "name"'
    ': "BAT-003", "severity": "error", "type": "custom"}, {"description": "Season must be between 2008 an'
    'd the current year", "dimension": "conformity", "engine": "fbl-dq-engine", "implementation": {"actio'
    'n": "quarantine", "columns": ["season"], "expression": "2008 AND year(current_date())", "owner": "Ba'
    'seball Operations", "rule_type": "range"}, "name": "BAT-004", "severity": "error", "type": "custom"}'
    ', {"description": "Games played must be between 0 and 162", "dimension": "conformity", "engine": "fb'
    'l-dq-engine", "implementation": {"action": "quarantine", "columns": ["g"], "expression": "0 AND 162"'
    ', "owner": "Baseball Operations", "rule_type": "range"}, "name": "BAT-005", "severity": "error", "ty'
    'pe": "custom"}, {"description": "Business rule: hits cannot exceed at bats", "dimension": "accuracy"'
    ', "engine": "fbl-dq-engine", "implementation": {"action": "quarantine", "columns": ["h"], "expressio'
    'n": "h <= ab", "owner": "Baseball Operations", "rule_type": "custom"}, "name": "BAT-006", "severity"'
    ': "error", "type": "custom"}, {"description": "Business rule: home runs cannot exceed hits", "dimens'
    'ion": "accuracy", "engine": "fbl-dq-engine", "implementation": {"action": "quarantine", "columns": ['
    '"hr"], "expression": "hr <= h", "owner": "Baseball Operations", "rule_type": "custom"}, "name": "BAT'
    '-007", "severity": "error", "type": "custom"}, {"description": "Business rule: extra-base hits canno'
    't exceed total hits", "dimension": "accuracy", "engine": "fbl-dq-engine", "implementation": {"action'
    '": "quarantine", "columns": ["h"], "expression": "doubles + triples + hr <= h", "owner": "Baseball O'
    'perations", "rule_type": "custom"}, "name": "BAT-008", "severity": "error", "type": "custom"}, {"des'
    'cription": "Business rule: no more than 6 at bats per game played", "dimension": "accuracy", "engine'
    '": "fbl-dq-engine", "implementation": {"action": "quarantine", "columns": ["ab"], "expression": "ab '
    '<= g * 6", "owner": "Baseball Operations", "rule_type": "custom"}, "name": "BAT-009", "severity": "e'
    'rror", "type": "custom"}, {"description": "Business rule: counting stats cannot be negative", "dimen'
    'sion": "accuracy", "engine": "fbl-dq-engine", "implementation": {"action": "quarantine", "columns": '
    '["ab"], "expression": "ab >= 0 AND h >= 0 AND doubles >= 0 AND triples >= 0 AND hr >= 0 AND rbi >= 0'
    ' AND bb >= 0 AND so >= 0 AND sb >= 0", "owner": "Baseball Operations", "rule_type": "custom"}, "name'
    '": "BAT-010", "severity": "error", "type": "custom"}, {"description": "player_id must exist in the c'
    'lean player table", "dimension": "consistency", "engine": "fbl-dq-engine", "implementation": {"actio'
    'n": "quarantine", "columns": ["player_id"], "expression": "silver.fbl_player.player_id", "owner": "B'
    'aseball Operations", "rule_type": "referential"}, "name": "BAT-011", "severity": "error", "type": "c'
    'ustom"}, {"description": "team_id must exist in the clean team table", "dimension": "consistency", "'
    'engine": "fbl-dq-engine", "implementation": {"action": "quarantine", "columns": ["team_id"], "expres'
    'sion": "silver.fbl_team.team_id", "owner": "Baseball Operations", "rule_type": "referential"}, "name'
    '": "BAT-012", "severity": "error", "type": "custom"}]}], "status": "active", "tags": ["fbl", "silver'
    '"], "version": "1.0.0"}, "id": "fbl.silver.batting"}, {"checksum": "ccaf03004ed81f90", "contract": {'
    '"apiVersion": "v3.1.0", "customProperties": [{"property": "layer", "value": "silver"}, {"property": '
    '"owner", "value": "Data Engineering"}, {"property": "dependsOn", "value": "fbl.source.player"}], "da'
    'taProduct": "fbl", "description": {"limitations": "Rejected rows are NOT in this table; they are in '
    'governance.dq_quarantine with the reason. Optional columns may still be NULL.", "purpose": "Typed, c'
    'leaned, deduplicated player rows. Every row here has passed every quarantine rule below.", "usage": '
    '"Internal layer for data engineering. Consumers read gold or the semantic layer."}, "domain": "baseb'
    'all_operations", "id": "fbl.silver.player", "kind": "DataContract", "name": "FBL player (silver)", "'
    'schema": [{"customProperties": [{"property": "entity", "value": "player"}], "description": "Clean pl'
    'ayer records.", "logicalType": "object", "name": "fbl_player", "physicalName": "silver.fbl_player", '
    '"physicalType": "table", "properties": [{"customProperties": [{"property": "standardize", "value": "'
    'upper"}], "description": "Player identifier, e.g. P001. Primary key.", "logicalType": "string", "nam'
    'e": "player_id", "physicalType": "STRING", "primaryKey": true, "primaryKeyPosition": 1, "required": '
    'true}, {"description": "Given name.", "logicalType": "string", "name": "first_name", "physicalType":'
    ' "STRING", "required": true}, {"description": "Family name.", "logicalType": "string", "name": "last'
    '_name", "physicalType": "STRING", "required": true}, {"classification": "personal", "description": "'
    'Date of birth. Personal data. Player must be at least 16.", "logicalType": "date", "name": "birth_da'
    'te", "physicalType": "DATE", "required": true}, {"customProperties": [{"property": "standardize", "v'
    'alue": "upper"}], "description": "Batting side: L, R or B (switch).", "logicalType": "string", "name'
    '": "bats", "physicalType": "STRING", "required": true}, {"customProperties": [{"property": "standard'
    'ize", "value": "upper"}], "description": "Throwing hand: L or R.", "logicalType": "string", "name": '
    '"throws", "physicalType": "STRING", "required": true}, {"customProperties": [{"property": "standardi'
    'ze", "value": "upper"}], "description": "Team the player belongs to.", "logicalType": "string", "nam'
    'e": "team_id", "physicalType": "STRING", "relationships": [{"to": "fbl_team.team_id", "type": "forei'
    'gnKey"}], "required": true}, {"customProperties": [{"property": "standardize", "value": "upper"}], "'
    'description": "Position code: P, C, 1B, 2B, 3B, SS, LF, CF, RF or DH.", "logicalType": "string", "na'
    'me": "position", "physicalType": "STRING", "required": true}, {"description": "Height in inches (60 '
    'to 84).", "logicalType": "integer", "name": "height_in", "physicalType": "INT", "required": true}, {'
    '"description": "Weight in pounds (120 to 350 expected).", "logicalType": "integer", "name": "weight_'
    'lb", "physicalType": "INT", "required": true}, {"description": "Optional nickname. Shown as \'None\' i'
    'n gold.", "logicalType": "string", "name": "nickname", "physicalType": "STRING", "required": false}]'
    ', "quality": [{"description": "Identical duplicate player rows are removed automatically (first copy'
    ' kept)", "dimension": "uniqueness", "engine": "fbl-dq-engine", "implementation": {"action": "dedupe"'
    ', "columns": "*", "expression": "", "owner": "Data Engineering", "rule_type": "exact_duplicate"}, "n'
    'ame": "PLR-001", "severity": "warning", "type": "custom"}, {"description": "player_id must identify '
    'exactly one player. Conflicting duplicates are rejected for human review", "dimension": "uniqueness"'
    ', "engine": "fbl-dq-engine", "implementation": {"action": "quarantine", "columns": ["player_id"], "e'
    'xpression": "", "owner": "Data Engineering", "rule_type": "unique_strict"}, "name": "PLR-002", "seve'
    'rity": "error", "type": "custom"}, {"description": "Required player fields must be present", "dimens'
    'ion": "completeness", "engine": "fbl-dq-engine", "implementation": {"action": "quarantine", "columns'
    '": ["player_id", "first_name", "last_name", "birth_date", "bats", "throws", "team_id", "position", "'
    'height_in", "weight_lb"], "expression": "", "owner": "Data Engineering", "rule_type": "not_null"}, "'
    'name": "PLR-003", "severity": "error", "type": "custom"}, {"description": "Bats must be L, R or B", '
    '"dimension": "conformity", "engine": "fbl-dq-engine", "implementation": {"action": "quarantine", "co'
    'lumns": ["bats"], "expression": "\'L\',\'R\',\'B\'", "owner": "Baseball Operations", "rule_type": "allowed'
    '_values"}, "name": "PLR-004", "severity": "error", "type": "custom"}, {"description": "Throws must b'
    'e L or R", "dimension": "conformity", "engine": "fbl-dq-engine", "implementation": {"action": "quara'
    'ntine", "columns": ["throws"], "expression": "\'L\',\'R\'", "owner": "Baseball Operations", "rule_type":'
    ' "allowed_values"}, "name": "PLR-005", "severity": "error", "type": "custom"}, {"description": "Posi'
    'tion must be a valid code", "dimension": "conformity", "engine": "fbl-dq-engine", "implementation": '
    '{"action": "quarantine", "columns": ["position"], "expression": "\'P\',\'C\',\'1B\',\'2B\',\'3B\',\'SS\',\'LF\',\'C'
    'F\',\'RF\',\'DH\'", "owner": "Baseball Operations", "rule_type": "allowed_values"}, "name": "PLR-006", "s'
    'everity": "error", "type": "custom"}, {"description": "Height must be between 60 and 84 inches", "di'
    'mension": "conformity", "engine": "fbl-dq-engine", "implementation": {"action": "quarantine", "colum'
    'ns": ["height_in"], "expression": "60 AND 84", "owner": "Baseball Operations", "rule_type": "range"}'
    ', "name": "PLR-007", "severity": "error", "type": "custom"}, {"description": "Weight outside 120-350'
    ' lb is unusual: logged, row kept", "dimension": "conformity", "engine": "fbl-dq-engine", "implementa'
    'tion": {"action": "warn", "columns": ["weight_lb"], "expression": "120 AND 350", "owner": "Baseball '
    'Operations", "rule_type": "range"}, "name": "PLR-008", "severity": "warning", "type": "custom"}, {"d'
    'escription": "Business rule: a player must be at least 16 years old (also catches future birth dates'
    ')", "dimension": "accuracy", "engine": "fbl-dq-engine", "implementation": {"action": "quarantine", "'
    'columns": ["birth_date"], "expression": "months_between(current_date(), birth_date) >= 16 * 12", "ow'
    'ner": "Baseball Operations", "rule_type": "custom"}, "name": "PLR-009", "severity": "error", "type":'
    ' "custom"}, {"description": "team_id must exist in the clean team table", "dimension": "consistency"'
    ', "engine": "fbl-dq-engine", "implementation": {"action": "quarantine", "columns": ["team_id"], "exp'
    'ression": "silver.fbl_team.team_id", "owner": "Baseball Operations", "rule_type": "referential"}, "n'
    'ame": "PLR-010", "severity": "error", "type": "custom"}, {"description": "bats was trimmed/upper-cas'
    'ed (auto-corrected)", "dimension": "conformity", "engine": "fbl-dq-engine", "implementation": {"acti'
    'on": "standardize", "columns": ["bats"], "expression": "", "owner": "Data Engineering", "rule_type":'
    ' "standardized"}, "name": "PLR-011", "severity": "warning", "type": "custom"}]}], "status": "active"'
    ', "tags": ["fbl", "silver"], "version": "1.0.0"}, "id": "fbl.silver.player"}, {"checksum": "201bd260'
    '6ccdffa1", "contract": {"apiVersion": "v3.1.0", "customProperties": [{"property": "layer", "value": '
    '"silver"}, {"property": "owner", "value": "Data Engineering"}, {"property": "dependsOn", "value": "f'
    'bl.source.team"}], "dataProduct": "fbl", "description": {"limitations": "Rejected rows are NOT in th'
    'is table; they are in governance.dq_quarantine with the reason. Optional columns may still be NULL."'
    ', "purpose": "Typed, cleaned, deduplicated team rows. Every row here has passed every quarantine rul'
    'e below.", "usage": "Internal layer for data engineering. Consumers read gold or the semantic layer.'
    '"}, "domain": "baseball_operations", "id": "fbl.silver.team", "kind": "DataContract", "name": "FBL t'
    'eam (silver)", "schema": [{"customProperties": [{"property": "entity", "value": "team"}], "descripti'
    'on": "Clean team records.", "logicalType": "object", "name": "fbl_team", "physicalName": "silver.fbl'
    '_team", "physicalType": "table", "properties": [{"customProperties": [{"property": "standardize", "v'
    'alue": "upper"}], "description": "Team identifier, e.g. T01. Primary key.", "logicalType": "string",'
    ' "name": "team_id", "physicalType": "STRING", "primaryKey": true, "primaryKeyPosition": 1, "required'
    '": true}, {"description": "Full team name.", "logicalType": "string", "name": "team_name", "physical'
    'Type": "STRING", "required": true}, {"description": "Home city.", "logicalType": "string", "name": "'
    'city", "physicalType": "STRING", "required": true}, {"customProperties": [{"property": "standardize"'
    ', "value": "initcap"}], "description": "League: North or South.", "logicalType": "string", "name": "'
    'league", "physicalType": "STRING", "required": true}, {"description": "Year the franchise was founde'
    'd (league started in 2000).", "logicalType": "integer", "name": "founded_year", "physicalType": "INT'
    '", "required": true}, {"description": "Home ballpark.", "logicalType": "string", "name": "home_park"'
    ', "physicalType": "STRING", "required": true}, {"description": "Payroll cap in millions of USD. Must'
    ' be positive.", "logicalType": "number", "name": "payroll_cap_musd", "physicalType": "DOUBLE", "requ'
    'ired": true}, {"description": "Team mascot. Optional: shown as \'None\' in gold.", "logicalType": "str'
    'ing", "name": "mascot", "physicalType": "STRING", "required": false}], "quality": [{"description": "'
    'Identical duplicate team rows are removed automatically (first copy kept)", "dimension": "uniqueness'
    '", "engine": "fbl-dq-engine", "implementation": {"action": "dedupe", "columns": "*", "expression": "'
    '", "owner": "Data Engineering", "rule_type": "exact_duplicate"}, "name": "TEAM-001", "severity": "wa'
    'rning", "type": "custom"}, {"description": "team_id must identify exactly one team. Conflicting dupl'
    'icates are rejected for human review", "dimension": "uniqueness", "engine": "fbl-dq-engine", "implem'
    'entation": {"action": "quarantine", "columns": ["team_id"], "expression": "", "owner": "Data Enginee'
    'ring", "rule_type": "unique_strict"}, "name": "TEAM-002", "severity": "error", "type": "custom"}, {"'
    'description": "Required team fields must be present", "dimension": "completeness", "engine": "fbl-dq'
    '-engine", "implementation": {"action": "quarantine", "columns": ["team_id", "team_name", "city", "le'
    'ague", "founded_year", "home_park", "payroll_cap_musd"], "expression": "", "owner": "Data Engineerin'
    'g", "rule_type": "not_null"}, "name": "TEAM-003", "severity": "error", "type": "custom"}, {"descript'
    'ion": "League must be North or South", "dimension": "conformity", "engine": "fbl-dq-engine", "implem'
    'entation": {"action": "quarantine", "columns": ["league"], "expression": "\'North\',\'South\'", "owner":'
    ' "Baseball Operations", "rule_type": "allowed_values"}, "name": "TEAM-004", "severity": "error", "ty'
    'pe": "custom"}, {"description": "Founded year must be between 2000 and the current year (league star'
    'ted in 2000)", "dimension": "conformity", "engine": "fbl-dq-engine", "implementation": {"action": "q'
    'uarantine", "columns": ["founded_year"], "expression": "2000 AND year(current_date())", "owner": "Ba'
    'seball Operations", "rule_type": "range"}, "name": "TEAM-005", "severity": "error", "type": "custom"'
    '}, {"description": "Business rule: payroll cap must be a positive number of millions", "dimension": '
    '"accuracy", "engine": "fbl-dq-engine", "implementation": {"action": "quarantine", "columns": ["payro'
    'll_cap_musd"], "expression": "payroll_cap_musd > 0", "owner": "Baseball Finance", "rule_type": "cust'
    'om"}, "name": "TEAM-006", "severity": "error", "type": "custom"}, {"description": "League text was t'
    'rimmed/re-cased (auto-corrected)", "dimension": "conformity", "engine": "fbl-dq-engine", "implementa'
    'tion": {"action": "standardize", "columns": ["league"], "expression": "", "owner": "Data Engineering'
    '", "rule_type": "standardized"}, "name": "TEAM-007", "severity": "warning", "type": "custom"}, {"des'
    'cription": "team_id was trimmed/upper-cased (auto-corrected)", "dimension": "conformity", "engine": '
    '"fbl-dq-engine", "implementation": {"action": "standardize", "columns": ["team_id"], "expression": "'
    '", "owner": "Data Engineering", "rule_type": "standardized"}, "name": "TEAM-008", "severity": "warni'
    'ng", "type": "custom"}]}], "status": "active", "tags": ["fbl", "silver"], "version": "1.0.0"}, "id":'
    ' "fbl.silver.team"}, {"checksum": "aeee23d3850dcf1c", "contract": {"apiVersion": "v3.1.0", "customPr'
    'operties": [{"property": "layer", "value": "source"}, {"property": "owner", "value": "Baseball Opera'
    'tions (file owner)"}, {"property": "sourceSystem", "value": "league_office_csv"}], "dataProduct": "f'
    'bl", "description": {"limitations": "Values arrive as text and may be blank, duplicated, mis-cased o'
    'r invalid. Row-level validity is NOT promised here; silver enforces it.", "purpose": "The batting.cs'
    'v file the league office sends us. Promise: the listed columns are present and the file is not empty'
    '.", "usage": "Load as strings into bronze. Never clean values before bronze."}, "domain": "baseball_'
    'operations", "id": "fbl.source.batting", "kind": "DataContract", "name": "FBL batting file (as deliv'
    'ered)", "schema": [{"customProperties": [{"property": "entity", "value": "batting"}, {"property": "l'
    'andingFolder", "value": "batting"}, {"property": "fileName", "value": "batting.csv"}], "description"'
    ': "Raw batting.csv, all columns as delivered.", "logicalType": "object", "name": "fbl_batting", "phy'
    'sicalName": "bronze.fbl_batting", "physicalType": "table", "properties": [{"description": "Player. P'
    'art of the key. (raw text, present in the file header)", "logicalType": "string", "name": "player_id'
    '", "physicalType": "STRING", "required": true}, {"description": "Season year. Part of the key. (raw '
    'text, present in the file header)", "logicalType": "string", "name": "season", "physicalType": "STRI'
    'NG", "required": true}, {"description": "Team played for. Part of the key. (raw text, present in the'
    ' file header)", "logicalType": "string", "name": "team_id", "physicalType": "STRING", "required": tr'
    'ue}, {"description": "Games played (0 to 162). (raw text, present in the file header)", "logicalType'
    '": "string", "name": "g", "physicalType": "STRING", "required": true}, {"description": "At bats. (ra'
    'w text, present in the file header)", "logicalType": "string", "name": "ab", "physicalType": "STRING'
    '", "required": true}, {"description": "Hits. Cannot exceed at bats. (raw text, present in the file h'
    'eader)", "logicalType": "string", "name": "h", "physicalType": "STRING", "required": true}, {"descri'
    'ption": "Doubles. (raw text, present in the file header)", "logicalType": "string", "name": "doubles'
    '", "physicalType": "STRING", "required": true}, {"description": "Triples. (raw text, present in the '
    'file header)", "logicalType": "string", "name": "triples", "physicalType": "STRING", "required": tru'
    'e}, {"description": "Home runs. Cannot exceed hits. (raw text, present in the file header)", "logica'
    'lType": "string", "name": "hr", "physicalType": "STRING", "required": true}, {"description": "Runs b'
    'atted in. (raw text, present in the file header)", "logicalType": "string", "name": "rbi", "physical'
    'Type": "STRING", "required": true}, {"description": "Walks. (raw text, present in the file header)",'
    ' "logicalType": "string", "name": "bb", "physicalType": "STRING", "required": true}, {"description":'
    ' "Strikeouts. (raw text, present in the file header)", "logicalType": "string", "name": "so", "physi'
    'calType": "STRING", "required": true}, {"description": "Stolen bases. Cannot be negative. (raw text,'
    ' present in the file header)", "logicalType": "string", "name": "sb", "physicalType": "STRING", "req'
    'uired": true}], "quality": [{"description": "The delivered file must contain at least one row.", "di'
    'mension": "completeness", "metric": "rowCount", "mustBeGreaterThan": 0, "severity": "error", "type":'
    ' "library"}]}], "slaProperties": [{"element": "fbl_batting", "property": "frequency", "unit": "d", "'
    'value": 1}, {"element": "fbl_batting", "property": "timeOfAvailability", "unit": "UTC", "value": "06'
    ':00"}], "status": "active", "tags": ["fbl", "source"], "version": "1.0.0"}, "id": "fbl.source.battin'
    'g"}, {"checksum": "56f4102a7972bc34", "contract": {"apiVersion": "v3.1.0", "customProperties": [{"pr'
    'operty": "layer", "value": "source"}, {"property": "owner", "value": "Baseball Operations (file owne'
    'r)"}, {"property": "sourceSystem", "value": "league_office_csv"}], "dataProduct": "fbl", "descriptio'
    'n": {"limitations": "Values arrive as text and may be blank, duplicated, mis-cased or invalid. Row-l'
    'evel validity is NOT promised here; silver enforces it.", "purpose": "The players.csv file the leagu'
    'e office sends us. Promise: the listed columns are present and the file is not empty.", "usage": "Lo'
    'ad as strings into bronze. Never clean values before bronze."}, "domain": "baseball_operations", "id'
    '": "fbl.source.player", "kind": "DataContract", "name": "FBL player file (as delivered)", "schema": '
    '[{"customProperties": [{"property": "entity", "value": "player"}, {"property": "landingFolder", "val'
    'ue": "players"}, {"property": "fileName", "value": "players.csv"}], "description": "Raw players.csv,'
    ' all columns as delivered.", "logicalType": "object", "name": "fbl_player", "physicalName": "bronze.'
    'fbl_player", "physicalType": "table", "properties": [{"description": "Player identifier, e.g. P001. '
    'Primary key. (raw text, present in the file header)", "logicalType": "string", "name": "player_id", '
    '"physicalType": "STRING", "required": true}, {"description": "Given name. (raw text, present in the '
    'file header)", "logicalType": "string", "name": "first_name", "physicalType": "STRING", "required": '
    'true}, {"description": "Family name. (raw text, present in the file header)", "logicalType": "string'
    '", "name": "last_name", "physicalType": "STRING", "required": true}, {"description": "Date of birth.'
    ' Personal data. Player must be at least 16. (raw text, present in the file header)", "logicalType": '
    '"string", "name": "birth_date", "physicalType": "STRING", "required": true}, {"description": "Battin'
    'g side: L, R or B (switch). (raw text, present in the file header)", "logicalType": "string", "name"'
    ': "bats", "physicalType": "STRING", "required": true}, {"description": "Throwing hand: L or R. (raw '
    'text, present in the file header)", "logicalType": "string", "name": "throws", "physicalType": "STRI'
    'NG", "required": true}, {"description": "Team the player belongs to. (raw text, present in the file '
    'header)", "logicalType": "string", "name": "team_id", "physicalType": "STRING", "required": true}, {'
    '"description": "Position code: P, C, 1B, 2B, 3B, SS, LF, CF, RF or DH. (raw text, present in the fil'
    'e header)", "logicalType": "string", "name": "position", "physicalType": "STRING", "required": true}'
    ', {"description": "Height in inches (60 to 84). (raw text, present in the file header)", "logicalTyp'
    'e": "string", "name": "height_in", "physicalType": "STRING", "required": true}, {"description": "Wei'
    'ght in pounds (120 to 350 expected). (raw text, present in the file header)", "logicalType": "string'
    '", "name": "weight_lb", "physicalType": "STRING", "required": true}, {"description": "Optional nickn'
    'ame. Shown as \'None\' in gold. (raw text, present in the file header)", "logicalType": "string", "nam'
    'e": "nickname", "physicalType": "STRING", "required": true}], "quality": [{"description": "The deliv'
    'ered file must contain at least one row.", "dimension": "completeness", "metric": "rowCount", "mustB'
    'eGreaterThan": 0, "severity": "error", "type": "library"}]}], "slaProperties": [{"element": "fbl_pla'
    'yer", "property": "frequency", "unit": "d", "value": 1}, {"element": "fbl_player", "property": "time'
    'OfAvailability", "unit": "UTC", "value": "06:00"}], "status": "active", "tags": ["fbl", "source"], "'
    'version": "1.0.0"}, "id": "fbl.source.player"}, {"checksum": "826e7d9c0a2eb05f", "contract": {"apiVe'
    'rsion": "v3.1.0", "customProperties": [{"property": "layer", "value": "source"}, {"property": "owner'
    '", "value": "Baseball Operations (file owner)"}, {"property": "sourceSystem", "value": "league_offic'
    'e_csv"}], "dataProduct": "fbl", "description": {"limitations": "Values arrive as text and may be bla'
    'nk, duplicated, mis-cased or invalid. Row-level validity is NOT promised here; silver enforces it.",'
    ' "purpose": "The teams.csv file the league office sends us. Promise: the listed columns are present '
    'and the file is not empty.", "usage": "Load as strings into bronze. Never clean values before bronze'
    '."}, "domain": "baseball_operations", "id": "fbl.source.team", "kind": "DataContract", "name": "FBL '
    'team file (as delivered)", "schema": [{"customProperties": [{"property": "entity", "value": "team"},'
    ' {"property": "landingFolder", "value": "teams"}, {"property": "fileName", "value": "teams.csv"}], "'
    'description": "Raw teams.csv, all columns as delivered.", "logicalType": "object", "name": "fbl_team'
    '", "physicalName": "bronze.fbl_team", "physicalType": "table", "properties": [{"description": "Team '
    'identifier, e.g. T01. Primary key. (raw text, present in the file header)", "logicalType": "string",'
    ' "name": "team_id", "physicalType": "STRING", "required": true}, {"description": "Full team name. (r'
    'aw text, present in the file header)", "logicalType": "string", "name": "team_name", "physicalType":'
    ' "STRING", "required": true}, {"description": "Home city. (raw text, present in the file header)", "'
    'logicalType": "string", "name": "city", "physicalType": "STRING", "required": true}, {"description":'
    ' "League: North or South. (raw text, present in the file header)", "logicalType": "string", "name": '
    '"league", "physicalType": "STRING", "required": true}, {"description": "Year the franchise was found'
    'ed (league started in 2000). (raw text, present in the file header)", "logicalType": "string", "name'
    '": "founded_year", "physicalType": "STRING", "required": true}, {"description": "Home ballpark. (raw'
    ' text, present in the file header)", "logicalType": "string", "name": "home_park", "physicalType": "'
    'STRING", "required": true}, {"description": "Payroll cap in millions of USD. Must be positive. (raw '
    'text, present in the file header)", "logicalType": "string", "name": "payroll_cap_musd", "physicalTy'
    'pe": "STRING", "required": true}, {"description": "Team mascot. Optional: shown as \'None\' in gold. ('
    'raw text, present in the file header)", "logicalType": "string", "name": "mascot", "physicalType": "'
    'STRING", "required": true}], "quality": [{"description": "The delivered file must contain at least o'
    'ne row.", "dimension": "completeness", "metric": "rowCount", "mustBeGreaterThan": 0, "severity": "er'
    'ror", "type": "library"}]}], "slaProperties": [{"element": "fbl_team", "property": "frequency", "uni'
    't": "d", "value": 1}, {"element": "fbl_team", "property": "timeOfAvailability", "unit": "UTC", "valu'
    'e": "06:00"}], "status": "active", "tags": ["fbl", "source"], "version": "1.0.0"}, "id": "fbl.source'
    '.team"}], "definitions": {"10_mv_batting.sql": "-- METRIC VIEW = governed semantic object. Define ea'
    'ch KPI ONCE here; every dashboard, notebook, Genie space\\n-- and BI tool then gets the same answer. '
    'Query with: SELECT dim, MEASURE(measure) FROM view GROUP BY dim.\\n-- Key idea: measures are aggregat'
    'ions evaluated at QUERY time at whatever grain the caller groups by.\\nCREATE OR REPLACE VIEW {{catal'
    'og}}.semantic.mv_batting\\nWITH METRICS\\nLANGUAGE YAML\\nAS $$\\nversion: 1.1\\ncomment: \\"Batting metri'
    'cs. Source grain is player-season-stint. Ratios are computed from summed components, never averaged.'
    '\\"\\nsource: {{catalog}}.gold.fact_batting_season\\n\\njoins:\\n  - name: player\\n    source: {{catalog}'
    "}.gold.dim_player\\n    'on': source.player_key = player.player_key\\n    rely:\\n      at_most_one_mat"
    "ch: true\\n  - name: team\\n    source: {{catalog}}.gold.dim_team_season\\n    'on': source.team_season"
    '_key = team.team_season_key\\n    rely:\\n      at_most_one_match: true\\n  - name: calendar\\n    sourc'
    "e: {{catalog}}.gold.dim_season\\n    'on': source.season = calendar.season\\n    rely:\\n      at_most_"
    'one_match: true\\n\\nfields:\\n  - name: season\\n    expr: source.season\\n    display_name: \\"Season\\"\\'
    'n    synonyms: [\\"year\\"]\\n  - name: decade\\n    expr: calendar.decade\\n    display_name: \\"Decade\\"'
    '\\n  - name: baseball_era\\n    expr: calendar.era\\n    display_name: \\"Baseball Era\\"\\n    comment: \\'
    '"Business-defined era classification, see glossary term Era.\\"\\n  - name: player_name\\n    expr: pla'
    'yer.full_name\\n    display_name: \\"Player\\"\\n    synonyms: [\\"batter\\", \\"hitter\\", \\"name\\"]\\n  - n'
    'ame: batting_hand\\n    expr: player.batting_hand\\n    display_name: \\"Bats\\"\\n    synonyms: [\\"hande'
    'dness\\"]\\n  - name: birth_country\\n    expr: player.birth_country\\n    display_name: \\"Birth Country'
    '\\"\\n  - name: team_name\\n    expr: team.team_name\\n    display_name: \\"Team\\"\\n  - name: franchise\\n'
    '    expr: team.franchise_name\\n    display_name: \\"Franchise\\"\\n  - name: league\\n    expr: team.lea'
    'gue_id\\n    display_name: \\"League\\"\\n  - name: division\\n    expr: team.division_id\\n    display_na'
    'me: \\"Division\\"\\n\\nmeasures:\\n  - name: games\\n    expr: SUM(g)\\n    display_name: \\"Games\\"\\n  - n'
    'ame: at_bats\\n    expr: SUM(ab)\\n    display_name: \\"At Bats\\"\\n    synonyms: [\\"AB\\"]\\n  - name: pl'
    'ate_appearances\\n    expr: SUM(plate_appearances)\\n    display_name: \\"Plate Appearances\\"\\n    syno'
    'nyms: [\\"PA\\"]\\n  - name: runs\\n    expr: SUM(r)\\n    display_name: \\"Runs\\"\\n  - name: hits\\n    ex'
    'pr: SUM(h)\\n    display_name: \\"Hits\\"\\n    synonyms: [\\"H\\"]\\n  - name: home_runs\\n    expr: SUM(hr'
    ')\\n    display_name: \\"Home Runs\\"\\n    synonyms: [\\"HR\\", \\"homers\\", \\"dingers\\"]\\n  - name: rbi\\n'
    '    expr: SUM(rbi)\\n    display_name: \\"RBI\\"\\n    synonyms: [\\"runs batted in\\"]\\n  - name: stolen_'
    'bases\\n    expr: SUM(sb)\\n    display_name: \\"Stolen Bases\\"\\n    synonyms: [\\"SB\\", \\"steals\\"]\\n  '
    '- name: walks\\n    expr: SUM(bb)\\n    display_name: \\"Walks\\"\\n    synonyms: [\\"BB\\", \\"bases on bal'
    'ls\\"]\\n  - name: strikeouts\\n    expr: SUM(so)\\n    display_name: \\"Strikeouts\\"\\n    synonyms: [\\"S'
    'O\\", \\"K\\"]\\n  - name: batting_avg\\n    expr: SUM(h) / NULLIF(SUM(ab), 0)\\n    display_name: \\"Batti'
    'ng Average\\"\\n    synonyms: [\\"AVG\\", \\"BA\\"]\\n    comment: \\"Hits divided by at bats. Never average'
    ' player averages: always divide summed hits by summed at bats.\\"\\n  - name: on_base_pct\\n    expr: ('
    'COALESCE(SUM(h), 0) + COALESCE(SUM(bb), 0) + COALESCE(SUM(hbp), 0)) / NULLIF(COALESCE(SUM(ab), 0) + '
    'COALESCE(SUM(bb), 0) + COALESCE(SUM(hbp), 0) + COALESCE(SUM(sf), 0), 0)\\n    display_name: \\"On-Base'
    ' Percentage\\"\\n    synonyms: [\\"OBP\\"]\\n  - name: slugging_pct\\n    expr: SUM(total_bases) / NULLIF('
    'SUM(ab), 0)\\n    display_name: \\"Slugging Percentage\\"\\n    synonyms: [\\"SLG\\"]\\n  - name: ops\\n    '
    'expr: MEASURE(on_base_pct) + MEASURE(slugging_pct)\\n    display_name: \\"OPS\\"\\n    synonyms: [\\"on-b'
    'ase plus slugging\\"]\\n    comment: \\"Composed measure: built from two earlier measures with MEASURE('
    '). Change OBP once, OPS follows.\\"\\n  - name: isolated_power\\n    expr: MEASURE(slugging_pct) - MEAS'
    'URE(batting_avg)\\n    display_name: \\"Isolated Power\\"\\n    synonyms: [\\"ISO\\"]\\n  - name: home_run_'
    'rate\\n    expr: SUM(hr) * 1.0 / NULLIF(SUM(plate_appearances), 0)\\n    display_name: \\"Home Run Rate'
    '\\"\\n  - name: strikeout_rate\\n    expr: SUM(so) * 1.0 / NULLIF(SUM(plate_appearances), 0)\\n    displ'
    'ay_name: \\"Strikeout Rate\\"\\n    synonyms: [\\"K%\\"]\\n  - name: walk_rate\\n    expr: SUM(bb) * 1.0 / '
    'NULLIF(SUM(plate_appearances), 0)\\n    display_name: \\"Walk Rate\\"\\n    synonyms: [\\"BB%\\"]\\n  - nam'
    'e: stolen_base_success_rate\\n    expr: SUM(sb) * 1.0 / NULLIF(SUM(sb) + SUM(cs), 0)\\n    display_nam'
    'e: \\"Stolen Base Success Rate\\"\\n  - name: batters\\n    expr: COUNT(DISTINCT source.player_key)\\n   '
    ' display_name: \\"Distinct Batters\\"\\n    comment: \\"Non-additive: distinct counts cannot be summed a'
    'cross groups. The metric view handles that for you.\\"\\n  - name: home_runs_regular_stints\\n    expr:'
    ' SUM(hr) FILTER (WHERE ab >= 400)\\n    display_name: \\"Home Runs (400+ AB stints)\\"\\n    comment: \\"'
    'Filtered measure example: only rows with at least 400 at bats.\\"\\n  - name: cumulative_home_runs\\n  '
    '  expr: SUM(hr)\\n    display_name: \\"Cumulative Home Runs\\"\\n    comment: \\"Window measure: running '
    'total ordered by season. Group by season to see the curve.\\"\\n    window:\\n      - order: season\\n  '
    '      range: cumulative\\n$$;\\n", "20_mv_pitching.sql": "CREATE OR REPLACE VIEW {{catalog}}.semantic.'
    'mv_pitching\\nWITH METRICS\\nLANGUAGE YAML\\nAS $$\\nversion: 1.1\\ncomment: \\"Pitching metrics. Source g'
    'rain is player-season-stint. innings = ipouts / 3, so every rate uses outs, not decimal innings.\\"\\n'
    'source: {{catalog}}.gold.fact_pitching_season\\n\\njoins:\\n  - name: player\\n    source: {{catalog}}.g'
    "old.dim_player\\n    'on': source.player_key = player.player_key\\n    rely:\\n      at_most_one_match:"
    " true\\n  - name: team\\n    source: {{catalog}}.gold.dim_team_season\\n    'on': source.team_season_ke"
    'y = team.team_season_key\\n    rely:\\n      at_most_one_match: true\\n  - name: calendar\\n    source: '
    "{{catalog}}.gold.dim_season\\n    'on': source.season = calendar.season\\n    rely:\\n      at_most_one"
    '_match: true\\n\\nfields:\\n  - name: season\\n    expr: source.season\\n    display_name: \\"Season\\"\\n  '
    '  synonyms: [\\"year\\"]\\n  - name: decade\\n    expr: calendar.decade\\n    display_name: \\"Decade\\"\\n '
    ' - name: baseball_era\\n    expr: calendar.era\\n    display_name: \\"Baseball Era\\"\\n  - name: player_'
    'name\\n    expr: player.full_name\\n    display_name: \\"Pitcher\\"\\n    synonyms: [\\"player\\", \\"name\\"'
    ']\\n  - name: throws\\n    expr: player.throws\\n    display_name: \\"Throws\\"\\n  - name: team_name\\n   '
    ' expr: team.team_name\\n    display_name: \\"Team\\"\\n  - name: franchise\\n    expr: team.franchise_nam'
    'e\\n    display_name: \\"Franchise\\"\\n  - name: league\\n    expr: team.league_id\\n    display_name: \\"'
    'League\\"\\n\\nmeasures:\\n  - name: games\\n    expr: SUM(g)\\n    display_name: \\"Games\\"\\n  - name: gam'
    'es_started\\n    expr: SUM(gs)\\n    display_name: \\"Games Started\\"\\n  - name: wins\\n    expr: SUM(w)'
    '\\n    display_name: \\"Wins\\"\\n  - name: losses\\n    expr: SUM(l)\\n    display_name: \\"Losses\\"\\n  - '
    'name: saves\\n    expr: SUM(sv)\\n    display_name: \\"Saves\\"\\n  - name: complete_games\\n    expr: SUM'
    '(cg)\\n    display_name: \\"Complete Games\\"\\n  - name: shutouts\\n    expr: SUM(sho)\\n    display_name'
    ': \\"Shutouts\\"\\n  - name: strikeouts\\n    expr: SUM(so)\\n    display_name: \\"Strikeouts\\"\\n    synon'
    'yms: [\\"K\\", \\"SO\\"]\\n  - name: walks\\n    expr: SUM(bb)\\n    display_name: \\"Walks\\"\\n    synonyms:'
    ' [\\"BB\\"]\\n  - name: hits_allowed\\n    expr: SUM(h)\\n    display_name: \\"Hits Allowed\\"\\n  - name: h'
    'ome_runs_allowed\\n    expr: SUM(hr)\\n    display_name: \\"Home Runs Allowed\\"\\n  - name: earned_runs\\'
    'n    expr: SUM(er)\\n    display_name: \\"Earned Runs\\"\\n  - name: innings_pitched\\n    expr: SUM(ipou'
    'ts) / 3.0\\n    display_name: \\"Innings Pitched\\"\\n    synonyms: [\\"IP\\"]\\n  - name: era\\n    expr: 2'
    '7.0 * SUM(er) / NULLIF(SUM(ipouts), 0)\\n    display_name: \\"ERA\\"\\n    synonyms: [\\"earned run avera'
    'ge\\"]\\n    comment: \\"Earned runs per nine innings = 27 x ER / outs recorded.\\"\\n  - name: whip\\n   '
    ' expr: 3.0 * (SUM(h) + SUM(bb)) / NULLIF(SUM(ipouts), 0)\\n    display_name: \\"WHIP\\"\\n    synonyms: '
    '[\\"walks plus hits per inning\\"]\\n  - name: strikeouts_per_9\\n    expr: 27.0 * SUM(so) / NULLIF(SUM('
    'ipouts), 0)\\n    display_name: \\"Strikeouts per 9\\"\\n    synonyms: [\\"K/9\\"]\\n  - name: walks_per_9\\'
    'n    expr: 27.0 * SUM(bb) / NULLIF(SUM(ipouts), 0)\\n    display_name: \\"Walks per 9\\"\\n    synonyms:'
    ' [\\"BB/9\\"]\\n  - name: home_runs_per_9\\n    expr: 27.0 * SUM(hr) / NULLIF(SUM(ipouts), 0)\\n    displ'
    'ay_name: \\"Home Runs per 9\\"\\n    synonyms: [\\"HR/9\\"]\\n  - name: strikeout_to_walk_ratio\\n    expr:'
    ' SUM(so) * 1.0 / NULLIF(SUM(bb), 0)\\n    display_name: \\"Strikeout-to-Walk Ratio\\"\\n    synonyms: [\\'
    '"K/BB\\"]\\n  - name: win_pct\\n    expr: SUM(w) * 1.0 / NULLIF(SUM(w) + SUM(l), 0)\\n    display_name: '
    '\\"Win Percentage\\"\\n  - name: pitchers\\n    expr: COUNT(DISTINCT source.player_key)\\n    display_nam'
    'e: \\"Distinct Pitchers\\"\\n$$;\\n", "30_mv_team_season.sql": "CREATE OR REPLACE VIEW {{catalog}}.seman'
    'tic.mv_team_season\\nWITH METRICS\\nLANGUAGE YAML\\nAS $$\\nversion: 1.1\\ncomment: \\"Team-season results'
    ', run environment, attendance and payroll. Source grain is team-season. Payroll exists from 1985 onl'
    'y.\\"\\nsource: {{catalog}}.gold.fact_team_season\\n\\njoins:\\n  - name: team\\n    source: {{catalog}}.g'
    "old.dim_team_season\\n    'on': source.team_season_key = team.team_season_key\\n    rely:\\n      at_mo"
    "st_one_match: true\\n  - name: calendar\\n    source: {{catalog}}.gold.dim_season\\n    'on': source.se"
    'ason = calendar.season\\n    rely:\\n      at_most_one_match: true\\n\\nfields:\\n  - name: season\\n    e'
    'xpr: source.season\\n    display_name: \\"Season\\"\\n    synonyms: [\\"year\\"]\\n  - name: decade\\n    ex'
    'pr: calendar.decade\\n    display_name: \\"Decade\\"\\n  - name: baseball_era\\n    expr: calendar.era\\n '
    '   display_name: \\"Baseball Era\\"\\n  - name: team_name\\n    expr: team.team_name\\n    display_name: '
    '\\"Team\\"\\n  - name: franchise\\n    expr: team.franchise_name\\n    display_name: \\"Franchise\\"\\n  - n'
    'ame: league\\n    expr: team.league_id\\n    display_name: \\"League\\"\\n  - name: division\\n    expr: t'
    'eam.division_id\\n    display_name: \\"Division\\"\\n  - name: ballpark\\n    expr: team.park_name\\n    d'
    'isplay_name: \\"Ballpark\\"\\n  - name: won_world_series\\n    expr: team.won_world_series\\n    display_'
    'name: \\"Won World Series\\"\\n    synonyms: [\\"champion\\"]\\n\\nmeasures:\\n  - name: team_seasons\\n    e'
    'xpr: COUNT(1)\\n    display_name: \\"Team Seasons\\"\\n  - name: games\\n    expr: SUM(g)\\n    display_na'
    'me: \\"Games\\"\\n  - name: wins\\n    expr: SUM(w)\\n    display_name: \\"Wins\\"\\n    synonyms: [\\"W\\"]\\n'
    '  - name: losses\\n    expr: SUM(l)\\n    display_name: \\"Losses\\"\\n    synonyms: [\\"L\\"]\\n  - name: w'
    'in_pct\\n    expr: SUM(w) * 1.0 / NULLIF(SUM(w) + SUM(l), 0)\\n    display_name: \\"Win Percentage\\"\\n '
    '   synonyms: [\\"winning percentage\\", \\"WPCT\\"]\\n  - name: runs_scored\\n    expr: SUM(r)\\n    displa'
    'y_name: \\"Runs Scored\\"\\n  - name: runs_allowed\\n    expr: SUM(ra)\\n    display_name: \\"Runs Allowed'
    '\\"\\n  - name: run_differential\\n    expr: SUM(r) - SUM(ra)\\n    display_name: \\"Run Differential\\"\\n'
    '    synonyms: [\\"run diff\\"]\\n  - name: pythagorean_win_pct\\n    expr: POWER(SUM(r), 1.83) / NULLIF('
    'POWER(SUM(r), 1.83) + POWER(SUM(ra), 1.83), 0)\\n    display_name: \\"Pythagorean Win Percentage\\"\\n  '
    '  synonyms: [\\"expected win pct\\"]\\n    comment: \\"Expected win percentage from runs scored and allo'
    'wed (exponent 1.83).\\"\\n  - name: wins_over_expected\\n    expr: MEASURE(win_pct) - MEASURE(pythagore'
    'an_win_pct)\\n    display_name: \\"Win Pct Over Expected\\"\\n    synonyms: [\\"luck\\"]\\n  - name: attend'
    'ance\\n    expr: SUM(attendance)\\n    display_name: \\"Attendance\\"\\n  - name: avg_attendance_per_home'
    '_game\\n    expr: SUM(attendance) / NULLIF(SUM(g_home) FILTER (WHERE attendance IS NOT NULL), 0)\\n   '
    ' display_name: \\"Average Attendance per Home Game\\"\\n    comment: \\"Only counts home games where att'
    'endance was recorded, so missing data does not drag the average down.\\"\\n  - name: payroll\\n    expr'
    ': SUM(payroll_usd)\\n    display_name: \\"Payroll (USD)\\"\\n    comment: \\"Sum of player salaries. NULL'
    ' before 1985.\\"\\n  - name: cost_per_win\\n    expr: SUM(payroll_usd) / NULLIF(SUM(w) FILTER (WHERE pa'
    'yroll_usd IS NOT NULL), 0)\\n    display_name: \\"Payroll per Win (USD)\\"\\n    comment: \\"Wins in the '
    'denominator are restricted to seasons that have payroll, otherwise pre-1985 wins would flatter the r'
    'atio.\\"\\n$$;\\n", "mv_fbl_batting.sql": "-- FBL batting metrics for consumers. Governed by contract f'
    'bl.semantic.mv_fbl_batting.\\n-- OBP and OPS are \\"approx\\": the fictional league data has no hit-by-'
    'pitch or sacrifice-fly columns.\\nCREATE OR REPLACE VIEW {{catalog}}.semantic.mv_fbl_batting\\nWITH ME'
    'TRICS\\nLANGUAGE YAML\\nAS $$\\nversion: 1.1\\ncomment: \\"FBL batting metrics. Source grain is player-se'
    'ason-team. Ratios are computed from summed components, never averaged.\\"\\nsource: {{catalog}}.gold.f'
    "bl_fact_batting\\n\\njoins:\\n  - name: player\\n    source: {{catalog}}.gold.fbl_dim_player\\n    'on': "
    'source.player_id = player.player_id\\n    rely:\\n      at_most_one_match: true\\n  - name: team\\n    s'
    "ource: {{catalog}}.gold.fbl_dim_team\\n    'on': source.team_id = team.team_id\\n    rely:\\n      at_m"
    'ost_one_match: true\\n\\nfields:\\n  - name: season\\n    expr: source.season\\n    display_name: \\"Seaso'
    'n\\"\\n    synonyms: [\\"year\\"]\\n  - name: player_name\\n    expr: player.full_name\\n    display_name: '
    '\\"Player\\"\\n    synonyms: [\\"batter\\", \\"hitter\\"]\\n  - name: batting_hand\\n    expr: player.bats\\n '
    '   display_name: \\"Bats\\"\\n  - name: position\\n    expr: player.position\\n    display_name: \\"Positi'
    'on\\"\\n  - name: team_name\\n    expr: team.team_name\\n    display_name: \\"Team\\"\\n  - name: league\\n '
    '   expr: team.league\\n    display_name: \\"League\\"\\n  - name: city\\n    expr: team.city\\n    display'
    '_name: \\"City\\"\\n\\nmeasures:\\n  - name: games\\n    expr: SUM(g)\\n    display_name: \\"Games\\"\\n  - na'
    'me: at_bats\\n    expr: SUM(ab)\\n    display_name: \\"At Bats\\"\\n    synonyms: [\\"AB\\"]\\n  - name: hit'
    's\\n    expr: SUM(h)\\n    display_name: \\"Hits\\"\\n    synonyms: [\\"H\\"]\\n  - name: doubles\\n    expr:'
    ' SUM(doubles)\\n    display_name: \\"Doubles\\"\\n  - name: triples\\n    expr: SUM(triples)\\n    display'
    '_name: \\"Triples\\"\\n  - name: home_runs\\n    expr: SUM(hr)\\n    display_name: \\"Home Runs\\"\\n    syn'
    'onyms: [\\"HR\\", \\"homers\\"]\\n  - name: rbi\\n    expr: SUM(rbi)\\n    display_name: \\"RBI\\"\\n  - name:'
    ' walks\\n    expr: SUM(bb)\\n    display_name: \\"Walks\\"\\n    synonyms: [\\"BB\\"]\\n  - name: strikeouts'
    '\\n    expr: SUM(so)\\n    display_name: \\"Strikeouts\\"\\n    synonyms: [\\"SO\\", \\"K\\"]\\n  - name: stol'
    'en_bases\\n    expr: SUM(sb)\\n    display_name: \\"Stolen Bases\\"\\n    synonyms: [\\"SB\\"]\\n  - name: b'
    'atting_avg\\n    expr: SUM(h) / NULLIF(SUM(ab), 0)\\n    display_name: \\"Batting Average\\"\\n    synony'
    'ms: [\\"AVG\\", \\"BA\\"]\\n    comment: \\"Summed hits divided by summed at bats. Never average player av'
    'erages.\\"\\n  - name: on_base_pct_approx\\n    expr: (SUM(h) + SUM(bb)) / NULLIF(SUM(ab) + SUM(bb), 0)'
    '\\n    display_name: \\"On-Base Percentage (approx)\\"\\n    synonyms: [\\"OBP\\"]\\n  - name: slugging_pct'
    '\\n    expr: SUM(total_bases) / NULLIF(SUM(ab), 0)\\n    display_name: \\"Slugging Percentage\\"\\n    sy'
    'nonyms: [\\"SLG\\"]\\n  - name: ops_approx\\n    expr: MEASURE(on_base_pct_approx) + MEASURE(slugging_pc'
    't)\\n    display_name: \\"OPS (approx)\\"\\n    synonyms: [\\"OPS\\"]\\n  - name: home_run_rate\\n    expr: '
    'SUM(hr) * 1.0 / NULLIF(SUM(ab), 0)\\n    display_name: \\"Home Run Rate (per at bat)\\"\\n  - name: batt'
    'ers\\n    expr: COUNT(DISTINCT source.player_id)\\n    display_name: \\"Distinct Batters\\"\\n    comment'
    ': \\"Non-additive distinct count: do not sum across groups.\\"\\n$$;\\n", "mv_fbl_roster.sql": "-- FBL r'
    'oster metrics for consumers. Governed by contract fbl.semantic.mv_fbl_roster.\\nCREATE OR REPLACE VIE'
    'W {{catalog}}.semantic.mv_fbl_roster\\nWITH METRICS\\nLANGUAGE YAML\\nAS $$\\nversion: 1.1\\ncomment: \\"F'
    'BL roster metrics. Source grain is one row per player.\\"\\nsource: {{catalog}}.gold.fbl_dim_player\\n\\'
    "njoins:\\n  - name: team\\n    source: {{catalog}}.gold.fbl_dim_team\\n    'on': source.team_id = team."
    'team_id\\n    rely:\\n      at_most_one_match: true\\n\\nfields:\\n  - name: team_name\\n    expr: team.te'
    'am_name\\n    display_name: \\"Team\\"\\n  - name: league\\n    expr: team.league\\n    display_name: \\"Le'
    'ague\\"\\n  - name: position\\n    expr: source.position\\n    display_name: \\"Position\\"\\n  - name: bat'
    'ting_hand\\n    expr: source.bats\\n    display_name: \\"Bats\\"\\n  - name: throwing_hand\\n    expr: sou'
    'rce.throws\\n    display_name: \\"Throws\\"\\n\\nmeasures:\\n  - name: players\\n    expr: COUNT(1)\\n    di'
    'splay_name: \\"Players\\"\\n    synonyms: [\\"headcount\\", \\"roster size\\"]\\n  - name: avg_height_in\\n  '
    '  expr: AVG(height_in)\\n    display_name: \\"Average Height (in)\\"\\n  - name: avg_weight_lb\\n    expr'
    ': AVG(weight_lb)\\n    display_name: \\"Average Weight (lb)\\"\\n  - name: switch_hitters\\n    expr: COU'
    'NT(1) FILTER (WHERE source.bats = \'B\')\\n    display_name: \\"Switch Hitters\\"\\n  - name: teams\\n    e'
    'xpr: COUNT(DISTINCT source.team_id)\\n    display_name: \\"Distinct Teams\\"\\n$$;\\n"}}'
)

DEFAULT_RULES_CSV = (
    '"rule_id","layer","table_name","rule_type","columns","expression","action","description","owner","active"\n'
    '"TEAM-001","silver","team","exact_duplicate","*","","dedupe","Identical duplicate team rows are removed automatically (first copy kept)","Data Engineering","Y"\n'
    '"TEAM-002","silver","team","unique_strict","team_id","","quarantine","team_id must identify exactly one team. Conflicting duplicates are rejected for human review","Data Engineering","Y"\n'
    '"TEAM-003","silver","team","not_null","team_id,team_name,city,league,founded_year,home_park,payroll_cap_musd","","quarantine","Required team fields must be present","Data Engineering","Y"\n'
    '"TEAM-004","silver","team","allowed_values","league","\'North\',\'South\'","quarantine","League must be North or South","Baseball Operations","Y"\n'
    '"TEAM-005","silver","team","range","founded_year","2000 AND year(current_date())","quarantine","Founded year must be between 2000 and the current year (league started in 2000)","Baseball Operations","Y"\n'
    '"TEAM-006","silver","team","custom","payroll_cap_musd","payroll_cap_musd > 0","quarantine","Business rule: payroll cap must be a positive number of millions","Baseball Finance","Y"\n'
    '"TEAM-007","silver","team","standardized","league","","standardize","League text was trimmed/re-cased (auto-corrected)","Data Engineering","Y"\n'
    '"TEAM-008","silver","team","standardized","team_id","","standardize","team_id was trimmed/upper-cased (auto-corrected)","Data Engineering","Y"\n'
    '"PLR-001","silver","player","exact_duplicate","*","","dedupe","Identical duplicate player rows are removed automatically (first copy kept)","Data Engineering","Y"\n'
    '"PLR-002","silver","player","unique_strict","player_id","","quarantine","player_id must identify exactly one player. Conflicting duplicates are rejected for human review","Data Engineering","Y"\n'
    '"PLR-003","silver","player","not_null","player_id,first_name,last_name,birth_date,bats,throws,team_id,position,height_in,weight_lb","","quarantine","Required player fields must be present","Data Engineering","Y"\n'
    '"PLR-004","silver","player","allowed_values","bats","\'L\',\'R\',\'B\'","quarantine","Bats must be L, R or B","Baseball Operations","Y"\n'
    '"PLR-005","silver","player","allowed_values","throws","\'L\',\'R\'","quarantine","Throws must be L or R","Baseball Operations","Y"\n'
    '"PLR-006","silver","player","allowed_values","position","\'P\',\'C\',\'1B\',\'2B\',\'3B\',\'SS\',\'LF\',\'CF\',\'RF\',\'DH\'","quarantine","Position must be a valid code","Baseball Operations","Y"\n'
    '"PLR-007","silver","player","range","height_in","60 AND 84","quarantine","Height must be between 60 and 84 inches","Baseball Operations","Y"\n'
    '"PLR-008","silver","player","range","weight_lb","120 AND 350","warn","Weight outside 120-350 lb is unusual: logged, row kept","Baseball Operations","Y"\n'
    '"PLR-009","silver","player","custom","birth_date","months_between(current_date(), birth_date) >= 16 * 12","quarantine","Business rule: a player must be at least 16 years old (also catches future birth dates)","Baseball Operations","Y"\n'
    '"PLR-010","silver","player","referential","team_id","silver.fbl_team.team_id","quarantine","team_id must exist in the clean team table","Baseball Operations","Y"\n'
    '"PLR-011","silver","player","standardized","bats","","standardize","bats was trimmed/upper-cased (auto-corrected)","Data Engineering","Y"\n'
    '"BAT-001","silver","batting","exact_duplicate","*","","dedupe","Identical duplicate batting rows are removed automatically (first copy kept)","Data Engineering","Y"\n'
    '"BAT-002","silver","batting","unique_strict","player_id,season,team_id","","quarantine","One batting line per player, season and team. Conflicting duplicates are rejected for human review","Data Engineering","Y"\n'
    '"BAT-003","silver","batting","not_null","player_id,season,team_id,g,ab,h,doubles,triples,hr,rbi,bb,so,sb","","quarantine","Required batting fields must be present (non-numeric text becomes NULL and is caught here)","Data Engineering","Y"\n'
    '"BAT-004","silver","batting","range","season","2008 AND year(current_date())","quarantine","Season must be between 2008 and the current year","Baseball Operations","Y"\n'
    '"BAT-005","silver","batting","range","g","0 AND 162","quarantine","Games played must be between 0 and 162","Baseball Operations","Y"\n'
    '"BAT-006","silver","batting","custom","h","h <= ab","quarantine","Business rule: hits cannot exceed at bats","Baseball Operations","Y"\n'
    '"BAT-007","silver","batting","custom","hr","hr <= h","quarantine","Business rule: home runs cannot exceed hits","Baseball Operations","Y"\n'
    '"BAT-008","silver","batting","custom","h","doubles + triples + hr <= h","quarantine","Business rule: extra-base hits cannot exceed total hits","Baseball Operations","Y"\n'
    '"BAT-009","silver","batting","custom","ab","ab <= g * 6","quarantine","Business rule: no more than 6 at bats per game played","Baseball Operations","Y"\n'
    '"BAT-010","silver","batting","custom","ab","ab >= 0 AND h >= 0 AND doubles >= 0 AND triples >= 0 AND hr >= 0 AND rbi >= 0 AND bb >= 0 AND so >= 0 AND sb >= 0","quarantine","Business rule: counting stats cannot be negative","Baseball Operations","Y"\n'
    '"BAT-011","silver","batting","referential","player_id","silver.fbl_player.player_id","quarantine","player_id must exist in the clean player table","Baseball Operations","Y"\n'
    '"BAT-012","silver","batting","referential","team_id","silver.fbl_team.team_id","quarantine","team_id must exist in the clean team table","Baseball Operations","Y"\n'
    '"GLD-001","gold","dim_team","no_nulls","*","","fail","Gold guarantee: no NULL in any dim_team column","Data Engineering","Y"\n'
    '"GLD-002","gold","dim_player","no_nulls","*","","fail","Gold guarantee: no NULL in any dim_player column","Data Engineering","Y"\n'
    '"GLD-003","gold","fact_batting","no_nulls","*","","fail","Gold guarantee: no NULL in any fact_batting column","Data Engineering","Y"\n'
    '"GLD-004","gold","dim_team","unique","team_id","","fail","dim_team grain: one row per team_id","Data Engineering","Y"\n'
    '"GLD-005","gold","dim_player","unique","player_id","","fail","dim_player grain: one row per player_id","Data Engineering","Y"\n'
    '"GLD-006","gold","fact_batting","unique","player_id,season,team_id","","fail","fact_batting grain: one row per player, season, team","Data Engineering","Y"\n'
    '"GLD-007","gold","fact_batting","referential","player_id","gold.fbl_dim_player.player_id","fail","Every batting row must have a player in dim_player","Data Engineering","Y"\n'
    '"GLD-008","gold","fact_batting","referential","team_id","gold.fbl_dim_team.team_id","fail","Every batting row must have a team in dim_team","Data Engineering","Y"\n'
    '"GLD-009","gold","dim_player","referential","team_id","gold.fbl_dim_team.team_id","fail","Every player must belong to a team in dim_team","Data Engineering","Y"\n'
    '"GLD-010","gold","fact_batting","custom","h","h <= ab AND hr <= h AND ab <= g * 6","fail","Business rules re-checked on gold: hits, home runs and at bats are consistent","Data Engineering","Y"\n'
)


In [ ]:

import json
import os
from datetime import datetime

dbutils.widgets.text("catalog", "mlb_dev")
dbutils.widgets.dropdown("stage", "all", ["all", "setup", "contracts", "bronze", "silver", "gold", "semantic", "report"])
dbutils.widgets.text("run_id", "")
dbutils.widgets.dropdown("fail_on_rejects", "true", ["true", "false"])
dbutils.widgets.dropdown("rules_source", "contracts", ["contracts", "csv"])
dbutils.widgets.dropdown("include_semantic", "true", ["true", "false"])
dbutils.widgets.dropdown("reload_rules", "false", ["false", "true"])
dbutils.widgets.text("landing_dir", "")
dbutils.widgets.text("contracts_dir", "")

catalog = dbutils.widgets.get("catalog").strip()
stage = dbutils.widgets.get("stage")
run_id = dbutils.widgets.get("run_id").strip() or datetime.now().strftime("%Y%m%d-%H%M%S")
fail_on_rejects = dbutils.widgets.get("fail_on_rejects") == "true"
rules_source = dbutils.widgets.get("rules_source")
include_semantic = dbutils.widgets.get("include_semantic") == "true"
landing = dbutils.widgets.get("landing_dir").strip() or f"/Volumes/{catalog}/bronze/landing/fbl"
contracts_dir = dbutils.widgets.get("contracts_dir").strip() or f"{landing}/contracts"
spark.sql(f"USE CATALOG {catalog}")


def load_contract_sources():
    """Contracts you uploaded (bundle JSON or YAML) win; otherwise the copy built into this notebook."""
    found = discover_contracts(contracts_dir) if os.path.isdir(contracts_dir) else None
    if found:
        return found
    b = json.loads(EMBEDDED_BUNDLE_JSON)
    return contracts_from_bundle(b), b.get("definitions", {}), "built-in bundle (nothing found in " + contracts_dir + ")"


rules_path = f"{landing}/rules/dq_rules.csv"          # legacy CSV mode only
rules_text = open(rules_path).read() if os.path.exists(rules_path) else DEFAULT_RULES_CSV
print(f"catalog={catalog} stage={stage} run_id={run_id} rules_source={rules_source}")

In [ ]:
failure = None
try:
    if stage in ("all", "setup"):
        stage_setup(spark, rules_text if rules_source == "csv" else None, dbutils.widgets.get("reload_rules") == "true")
    if stage in ("all", "contracts") and rules_source == "contracts":
        contracts, definitions, origin = load_contract_sources()
        stage_contracts(spark, run_id, contracts, definitions, origin)
    if stage in ("all", "bronze"):
        stage_bronze(spark, landing, run_id)
    if stage in ("all", "silver"):
        stage_silver(spark, run_id)
    if stage in ("all", "gold"):
        stage_gold(spark, run_id)
    if stage == "semantic" or (stage == "all" and include_semantic):
        _, definitions, _ = load_contract_sources()
        stage_semantic(spark, run_id, catalog, definitions)
except Exception as e:                       # keep going so the report still gets written
    failure = e
    print("STAGE FAILED:", e)

In [ ]:
if stage in ("all", "report"):
    report = build_report(spark, run_id)
    print(report["text"])
    displayHTML(report["html"])
    try:
        out_dir = os.path.dirname(landing) + "/reports"
        os.makedirs(out_dir, exist_ok=True)
        with open(f"{out_dir}/dq_report_{run_id}.html", "w") as fh:
            fh.write(report["html"])
        print(f"report file: {out_dir}/dq_report_{run_id}.html")
    except Exception as e:                   # reporting must never hide the real result
        print("could not write report file:", e)
    if failure is None and fail_on_rejects and (report["rejected"] > 0 or report["failed_rules"] > 0 or report["contract_failures"] > 0):
        failure = DQFailure("KICK BACK: " + report["text"].splitlines()[0])

if failure is not None:
    raise failure